# IIDS67692 Computational Techniques for Multi-modal Data
# WSI-Bench GPT-2 + LoRA with Perturbation-Aware Semantic Hallucination Entropy

This notebook adapts the PathVQA PA-SHE experiment to **WSI-Bench**. It uses
the WSI patch-feature files referenced by the official annotations rather than
trying to load a gigapixel slide as one ordinary image.

In [ ]:
!nvidia-smi

### Dependency environment

In [ ]:
# Read-only dependency preflight. Install/upgrade on the login node, then
# launch nbconvert as a new process; never mutate this shared environment here.
from importlib.metadata import PackageNotFoundError, version as package_version
from packaging.specifiers import SpecifierSet
from packaging.version import Version

HF_STACK_REQUIREMENTS = {
    "transformers": ">=5.9,<6",
    "tokenizers": ">=0.22,<1",
    "peft": ">=0.20,<1",
    "accelerate": ">=1.13,<2",
    "datasets": ">=5.0,<6",
    "evaluate": ">=0.4.6,<1",
    "sentence-transformers": ">=5.6,<7",
    "huggingface-hub": ">=1.5,<2",
    "regex": ">=2026.1.15",
}
HF_STACK_INSTALL_COMMAND = (
    'python -m pip install --upgrade '
    '"transformers>=5.9,<6" "tokenizers>=0.22,<1" '
    '"peft>=0.20,<1" "accelerate>=1.13,<2" '
    '"datasets>=5.0,<6" "evaluate>=0.4.6,<1" '
    '"sentence-transformers>=5.6,<7" '
    '"huggingface-hub>=1.5,<2" "regex>=2026.1.15"'
)
TRANSFORMERS_REPAIR_COMMAND = (
    'python -m pip install --no-cache-dir --force-reinstall --no-deps '
    '"transformers==5.9.0"'
)

try:
    actual_hf_stack = {
        name: package_version(name) for name in HF_STACK_REQUIREMENTS
    }
    torch_version = package_version("torch")
except PackageNotFoundError as error:
    raise RuntimeError(
        f"Missing dependency: {error}. Activate the project .venv and run: "
        f"{HF_STACK_INSTALL_COMMAND}"
    ) from error

incompatible = {
    name: {"required": requirement, "actual": actual_hf_stack[name]}
    for name, requirement in HF_STACK_REQUIREMENTS.items()
    if Version(actual_hf_stack[name]) not in SpecifierSet(requirement)
}
if incompatible:
    raise RuntimeError(
        f"Incompatible Hugging Face stack: {incompatible}. "
        f"Activate the project .venv and run: {HF_STACK_INSTALL_COMMAND}"
    )
if Version(torch_version) < Version("2.4"):
    raise RuntimeError(
        f"torch {torch_version} is too old; Transformers 5.9 requires "
        "torch >= 2.4. Install a CUDA-matched PyTorch build outside the notebook."
    )

try:
    import datasets as _datasets
    import evaluate as _evaluate
    import torchvision as _torchvision
    from accelerate.utils.memory import clear_device_cache as _clear_device_cache
    from peft import LoraConfig as _LoraConfig, TaskType as _TaskType
    from peft import get_peft_model as _get_peft_model
    from sentence_transformers import SentenceTransformer as _SentenceTransformer
    from transformers import AutoModelForSequenceClassification as _AutoNLI
    from transformers import AutoTokenizer as _AutoTokenizer
    from transformers import EncoderDecoderCache as _EncoderDecoderCache
    from transformers import GPT2LMHeadModel as _GPT2LMHeadModel
    from transformers import GPT2Tokenizer as _GPT2Tokenizer
except FileNotFoundError as error:
    missing_path = str(error)
    if "site-packages/transformers/" in missing_path:
        raise RuntimeError(
            "The Transformers installation is incomplete: package metadata exists, "
            f"but an installed source file is missing ({missing_path}). Run: "
            f"{TRANSFORMERS_REPAIR_COMMAND}"
        ) from error
    raise
except Exception as error:
    raise RuntimeError(
        "Package versions satisfy the supported ranges, but model imports failed. "
        "Run `python -m pip check` and verify torch/torchvision compatibility."
    ) from error

print({
    **actual_hf_stack,
    "torch": torch_version,
    "torchvision": package_version("torchvision"),
    "compatibility_preflight": "passed",
})
del (
    _datasets, _evaluate, _torchvision, _clear_device_cache,
    _LoraConfig, _TaskType, _get_peft_model, _SentenceTransformer,
    _AutoNLI, _AutoTokenizer, _EncoderDecoderCache,
    _GPT2LMHeadModel, _GPT2Tokenizer,
)

# Dataset: WSI-Bench annotations and local WSI patch features

Before running this cell:

1. Accept access to `Lucas-yuc/WSI-Bench` on Hugging Face and authenticate with
   `huggingface-cli login` (or export `HF_TOKEN`).
2. Download the complete separate `Lucas-yuc/WSIBench-pt` feature repository (or use
   a complete existing copy) and point `WSIBENCH_FEATURE_ROOT` to its root. Check the
   download size first with `hf download Lucas-yuc/WSIBench-pt --repo-type dataset
   --local-dir /path/to/features --dry-run`, then remove `--dry-run` to download.
   The annotation `image`
   values must resolve to `.pt`, `.pth`, `.npy`, `.npz`, `.h5`, or `.hdf5`
   feature files below that root.
3. Optionally set `WSIBENCH_ANNOTATION_DIR`; otherwise annotations are stored
   under `data/WSI-Bench/annotations` in the current project.

The public release has a training annotation file and an open-test file; a
local closed-test file is used when available. The published feature repository
does not cover every slide referenced by the current annotations, so the default
`published_subset` protocol keeps only questions whose released feature file is
available and prints complete coverage statistics. This must be reported as a
**published-feature subset**, not as the full official split. Set
`WSIBENCH_FEATURE_COVERAGE_MODE=strict` to require full coverage and fail on any
missing feature. WSI-Bench has no validation file. When enough official
training features exist, this notebook makes a deterministic **slide-level**
validation split from training data. If the small published feature subset has
fewer than two usable training slides, it instead creates a deterministic,
slide-disjoint train/validation/test holdout from all feature-supported
annotations. The fallback is labelled **derived published-feature holdout** and
must not be reported as an official WSI-Bench test result. Questions belonging
to held-out test slides are never used for training or model selection.

In [ ]:
import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

WSIBENCH_REPO_ID = "Lucas-yuc/WSI-Bench"
WSIBENCH_FEATURE_REPO_ID = "Lucas-yuc/WSIBench-pt"
WSIBENCH_ANNOTATION_DIR = Path(
    os.environ.get("WSIBENCH_ANNOTATION_DIR", "data/WSI-Bench/annotations")
).expanduser().resolve()
WSIBENCH_FEATURE_EXTENSIONS = {
    ".pt", ".pth", ".npy", ".npz", ".h5", ".hdf5"
}


def wsibench_find_feature_root():
    """Resolve an explicit or common project-local patch-feature root."""
    requested = os.environ.get("WSIBENCH_FEATURE_ROOT", "").strip()
    project_dir = Path.cwd().resolve()
    candidates = []
    if requested:
        candidates.append(Path(requested).expanduser())
    candidates.extend([
        WSIBENCH_ANNOTATION_DIR.parent / "features",
        WSIBENCH_ANNOTATION_DIR / "features",
        project_dir / "data" / "WSI-Bench" / "features",
        project_dir / "WSI-Bench" / "features",
        Path.home() / "dissertation_2026" / "data" / "WSI-Bench" / "features",
        Path.home() / "dissertation_2026" / "pa_she_wsibench" / "data" / "WSI-Bench" / "features",
    ])
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.is_dir() and any(
            path.is_file() and path.suffix.lower() in WSIBENCH_FEATURE_EXTENSIONS
            for path in candidate.rglob("*")
        ):
            return candidate

    checked_text = "\n  - ".join(str(path) for path in checked)
    requested_text = requested or "<not set>"
    raise FileNotFoundError(
        "WSI-Bench requires precomputed patch-feature files; the annotation "
        "download does not create them.\n"
        f"WSIBENCH_FEATURE_ROOT={requested_text!r}\n"
        f"Checked:\n  - {checked_text}\n"
        "Set the real directory before nbconvert, for example:\n"
        "  export WSIBENCH_FEATURE_ROOT=\"${HOME}/dissertation_2026/"
        "pa_she_wsibench/data/WSI-Bench/features\"\n"
        "Do not leave the placeholder /path/to/your/wsi/features."
    )


WSIBENCH_FEATURE_ROOT = wsibench_find_feature_root()
WSIBENCH_VALIDATION_FRACTION = float(
    os.environ.get("WSIBENCH_VALIDATION_FRACTION", "0.10")
)
WSIBENCH_DERIVED_TEST_FRACTION = float(
    os.environ.get("WSIBENCH_DERIVED_TEST_FRACTION", "0.20")
)
WSIBENCH_SPLIT_SEED = int(os.environ.get("WSIBENCH_SPLIT_SEED", "42"))
WSIBENCH_TEST_MODE = os.environ.get("WSIBENCH_TEST_MODE", "both").strip().lower()
WSIBENCH_FEATURE_COVERAGE_MODE = os.environ.get(
    "WSIBENCH_FEATURE_COVERAGE_MODE", "published_subset"
).strip().lower()

if not 0.0 < WSIBENCH_VALIDATION_FRACTION < 1.0:
    raise ValueError("WSIBENCH_VALIDATION_FRACTION must be between 0 and 1.")
if not 0.0 < WSIBENCH_DERIVED_TEST_FRACTION < 1.0:
    raise ValueError("WSIBENCH_DERIVED_TEST_FRACTION must be between 0 and 1.")
if WSIBENCH_TEST_MODE not in {"both", "open", "closed"}:
    raise ValueError("WSIBENCH_TEST_MODE must be: both, open, or closed.")
if WSIBENCH_FEATURE_COVERAGE_MODE not in {"strict", "published_subset"}:
    raise ValueError(
        "WSIBENCH_FEATURE_COVERAGE_MODE must be strict or published_subset."
    )
WSIBENCH_TEST_SCOPE_LABEL = (
    "official test"
    if WSIBENCH_FEATURE_COVERAGE_MODE == "strict"
    else "published-feature test subset"
)
WSIBENCH_SPLIT_PROTOCOL = "official annotation split"
WSIBENCH_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
WSIBENCH_FILES = {
    "train": "WSI-Bench-train.json",
    "open": "WSI-Bench-open-question.jsonl",
}
WSIBENCH_CLOSED_FILENAME = "WSI-Bench-close-question.jsonl"


def wsibench_annotation_path(filename, required=True):
    local_path = WSIBENCH_ANNOTATION_DIR / filename
    if local_path.exists():
        return local_path
    try:
        downloaded = hf_hub_download(
            repo_id=WSIBENCH_REPO_ID,
            repo_type="dataset",
            filename=filename,
            local_dir=WSIBENCH_ANNOTATION_DIR,
            token=os.environ.get("HF_TOKEN") or None,
        )
    except Exception as error:
        if not required:
            return None
        raise RuntimeError(
            f"Could not obtain gated WSI-Bench annotation {filename}. "
            "Accept the dataset conditions and run `huggingface-cli login`, "
            "or copy the file into WSIBENCH_ANNOTATION_DIR."
        ) from error
    return Path(downloaded)


def wsibench_read_records(path):
    path = Path(path)
    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as handle:
            rows = [json.loads(line) for line in handle if line.strip()]
        return [
            record
            for row in rows
            for record in (row if isinstance(row, list) else [row])
        ]
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("data", "records", "annotations", "items"):
            if isinstance(payload.get(key), list):
                return payload[key]
    raise ValueError(f"Unsupported annotation structure: {path}")


def wsibench_first(record, keys, default=None):
    for key in keys:
        value = record.get(key)
        if value is not None and str(value).strip():
            return value
    return default


def wsibench_scalar_text(value, preferred_keys=()):
    if value is None:
        return ""
    if isinstance(value, dict):
        for key in (*preferred_keys, "text", "value", "content", "answer"):
            if key in value:
                text = wsibench_scalar_text(value[key], preferred_keys)
                if text:
                    return text
        return ""
    if isinstance(value, (list, tuple)):
        for item in value:
            text = wsibench_scalar_text(item, preferred_keys)
            if text:
                return text
        return ""
    return str(value).strip()


def wsibench_clean_question(value):
    text = str(value or "")
    text = re.sub(r"<image(?:_placeholder)?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def wsibench_canonical_slide_id(image_ref, fallback):
    """Use the feature filename as the split key across all annotation files.

    WSI-Bench train/open/closed files expose different auxiliary ID fields.
    Those fields are not guaranteed to identify a WSI consistently and can
    collapse many records onto one apparent slide.  The image/feature path is
    the shared identifier and therefore the safe key for leakage checks.
    """
    name = Path(str(image_ref or "")).name.strip()
    if not name:
        return str(fallback)
    lower_name = name.lower()
    for extension in sorted(WSIBENCH_FEATURE_EXTENSIONS, key=len, reverse=True):
        if lower_name.endswith(extension):
            return name[:-len(extension)]
    return Path(name).stem or str(fallback)


def wsibench_conversation_pairs(record):
    conversation = wsibench_first(
        record, ("conversations", "conversation", "messages"), []
    )
    if not isinstance(conversation, list):
        return []
    pairs = []
    pending_question = None
    for message in conversation:
        if not isinstance(message, dict):
            continue
        role = str(wsibench_first(message, ("from", "role", "speaker"), "")).lower()
        value = wsibench_first(message, ("value", "content", "text"), "")
        if role in {"human", "user", "question"}:
            pending_question = wsibench_clean_question(value)
        elif role in {"gpt", "assistant", "answer"} and pending_question:
            pairs.append((pending_question, str(value).strip()))
            pending_question = None
    return pairs


def wsibench_infer_answer_type(question, answer, source_kind, record):
    if source_kind == "open":
        return "Open-ended"
    if source_kind == "closed":
        return "Closed-ended"
    declared = str(
        wsibench_first(record, ("answer_type", "question_type", "type", "mode"), "")
    ).lower()
    closed_markers = ("close", "multiple", "choice", "mcq", "true", "false", "yes/no")
    if any(marker in declared for marker in closed_markers):
        return "Closed-ended"
    question_lower = str(question).lower()
    answer_lower = str(answer).strip().lower()
    has_options = bool(re.search(r"(?:^|\s)[a-d][\).:]\s", question_lower))
    if has_options or answer_lower in {"yes", "no", "true", "false", "a", "b", "c", "d"}:
        return "Closed-ended"
    return "Open-ended"


def wsibench_normalize_records(raw_records, source_kind):
    normalized = []
    for raw_index, record in enumerate(raw_records):
        if not isinstance(record, dict):
            continue
        image_ref = wsibench_scalar_text(wsibench_first(
            record,
            (
                "image", "image_id", "image_name", "image_path",
                "feature", "feature_path", "wsi", "wsi_id",
                "wsi_name", "wsi_path", "slide", "slide_id",
                "slide_name", "slide_path",
            ),
            "",
        ), ("path", "file_name", "filename", "id"))
        slide_id = wsibench_canonical_slide_id(
            image_ref, f"{source_kind}-record-{raw_index}"
        )
        direct_question = wsibench_scalar_text(wsibench_first(
            record,
            ("question", "questions", "prompt", "instruction",
             "query", "text", "text_input", "input"),
            None,
        ), ("question", "prompt"))
        direct_answer = wsibench_scalar_text(wsibench_first(
            record,
            ("answer", "answers", "T-answer", "T_answer",
             "t-answer", "t_answer", "reference", "reference_answer",
             "response", "label", "gt_answer", "ground_truth",
             "gt", "text_output", "output", "target"),
            None,
        ), ("answer", "text"))
        pairs = wsibench_conversation_pairs(record)
        if direct_question and direct_answer:
            pairs.insert(0, (
                wsibench_clean_question(direct_question),
                direct_answer,
            ))
        for pair_index, (question, answer) in enumerate(pairs):
            if not image_ref or not question or not answer:
                continue
            normalized.append({
                "record_id": str(wsibench_first(
                    record, ("id", "question_id", "uid"),
                    f"{source_kind}-{raw_index}-{pair_index}",
                )),
                "image": image_ref,
                "slide_id": slide_id,
                "question": question,
                "answer": answer,
                "answer_type": wsibench_infer_answer_type(
                    question, answer, source_kind, record
                ),
                "source_file": source_kind,
            })
    if not normalized:
        sample_types = sorted({type(record).__name__ for record in raw_records[:20]})
        sample_keys = sorted({
            str(key)
            for record in raw_records[:20] if isinstance(record, dict)
            for key in record.keys()
        })
        raise ValueError(
            f"No usable question/answer records were parsed from {source_kind}. "
            f"Sample record types={sample_types}; keys={sample_keys}."
        )
    return normalized


annotation_paths = {
    kind: wsibench_annotation_path(filename, required=True)
    for kind, filename in WSIBENCH_FILES.items()
}
annotation_paths["closed"] = wsibench_annotation_path(
    WSIBENCH_CLOSED_FILENAME, required=False
)
wsibench_train_all = wsibench_normalize_records(
    wsibench_read_records(annotation_paths["train"]), "train"
)
wsibench_open_test = wsibench_normalize_records(
    wsibench_read_records(annotation_paths["open"]), "open"
)
if annotation_paths["closed"] is None:
    wsibench_closed_test = []
    warnings.warn(
        "Closed-test annotations were not found in the release; "
        "WSIBENCH_TEST_MODE='both' will evaluate the available open test."
    )
else:
    wsibench_closed_test = wsibench_normalize_records(
        wsibench_read_records(annotation_paths["closed"]), "closed"
    )


def wsibench_published_feature_keys():
    keys = set()
    for path in WSIBENCH_FEATURE_ROOT.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in WSIBENCH_FEATURE_EXTENSIONS:
            continue
        relative = path.relative_to(WSIBENCH_FEATURE_ROOT).as_posix()
        keys.update((relative, path.name, path.stem))
    return keys


WSIBENCH_PUBLISHED_FEATURE_KEYS = wsibench_published_feature_keys()


def wsibench_has_published_feature(record):
    reference = Path(str(record["image"]))
    reference_posix = reference.as_posix().lstrip("./")
    return any(
        key in WSIBENCH_PUBLISHED_FEATURE_KEYS
        for key in (reference_posix, reference.name, reference.stem)
    )


def wsibench_apply_feature_coverage(records, source_name):
    supported = [record for record in records if wsibench_has_published_feature(record)]
    full_slides = {record["slide_id"] for record in records}
    supported_slides = {record["slide_id"] for record in supported}
    return supported, {
        "annotation_source": source_name,
        "annotated_questions": len(records),
        "retained_questions": len(supported),
        "dropped_questions": len(records) - len(supported),
        "question_coverage_pct": (
            100.0 * len(supported) / len(records) if records else np.nan
        ),
        "annotated_slides": len(full_slides),
        "retained_slides": len(supported_slides),
        "dropped_slides": len(full_slides - supported_slides),
    }


if WSIBENCH_FEATURE_COVERAGE_MODE == "published_subset":
    coverage_rows = []
    wsibench_train_all, row = wsibench_apply_feature_coverage(
        wsibench_train_all, "train annotations"
    )
    coverage_rows.append(row)
    wsibench_open_test, row = wsibench_apply_feature_coverage(
        wsibench_open_test, "open-test annotations"
    )
    coverage_rows.append(row)
    wsibench_closed_test, row = wsibench_apply_feature_coverage(
        wsibench_closed_test, "closed-test annotations"
    )
    coverage_rows.append(row)
    wsibench_feature_coverage = pd.DataFrame(coverage_rows)
    display(wsibench_feature_coverage)
    warnings.warn(
        "Evaluation scope is the published-feature subset because the released "
        "feature repository does not cover every annotated slide. Report the "
        "coverage table with all results."
    )
else:
    wsibench_feature_coverage = pd.DataFrame()

if not wsibench_train_all:
    raise RuntimeError("No training questions have matching published features.")

if WSIBENCH_TEST_MODE == "open":
    wsibench_test = wsibench_open_test
elif WSIBENCH_TEST_MODE == "closed":
    if not wsibench_closed_test:
        raise RuntimeError(
            "WSIBENCH_TEST_MODE='closed' requires closed-test annotations "
            "with matching published feature files."
        )
    wsibench_test = wsibench_closed_test
else:
    wsibench_test = wsibench_open_test + wsibench_closed_test
if not wsibench_test:
    raise RuntimeError(
        "No selected test questions have matching published feature files."
    )

def wsibench_deduplicate_records(records):
    unique = []
    seen = set()
    for record in records:
        key = (
            record["slide_id"],
            re.sub(r"\s+", " ", record["question"].strip().lower()),
            re.sub(r"\s+", " ", record["answer"].strip().lower()),
        )
        if key not in seen:
            seen.add(key)
            unique.append(record)
    return unique


# First attempt the official annotation split.  The small published feature
# release may contain test slides but almost no official training slides.
# In that case, build a clearly labelled derived holdout from all supported
# annotations.  Splitting remains slide-disjoint, so no WSI appears in more
# than one partition.
test_slide_ids = {record["slide_id"] for record in wsibench_test}
development_pool = [
    record for record in wsibench_train_all
    if record["slide_id"] not in test_slide_ids
]
development_slide_ids = sorted({record["slide_id"] for record in development_pool})
rng = np.random.default_rng(WSIBENCH_SPLIT_SEED)

if len(development_slide_ids) < 2:
    if WSIBENCH_FEATURE_COVERAGE_MODE != "published_subset":
        raise RuntimeError(
            "The official split has fewer than two feature-supported "
            "development slides. Add official training features or use "
            "WSIBENCH_FEATURE_COVERAGE_MODE=published_subset."
        )

    combined_supported = wsibench_deduplicate_records(
        wsibench_train_all + wsibench_open_test + wsibench_closed_test
    )
    candidate_test_slides = sorted({
        record["slide_id"] for record in wsibench_test
    })
    all_supported_slides = {
        record["slide_id"] for record in combined_supported
    }
    if len(candidate_test_slides) < 3:
        raise RuntimeError(
            "At least three feature-supported slides are needed for the "
            "derived train/validation/test holdout. Add more .pt features."
        )

    rng.shuffle(candidate_test_slides)
    derived_test_count = min(
        len(candidate_test_slides) - 2,
        max(1, int(round(
            len(candidate_test_slides) * WSIBENCH_DERIVED_TEST_FRACTION
        ))),
    )
    test_slide_ids = set(candidate_test_slides[:derived_test_count])
    wsibench_test = [
        record for record in wsibench_test
        if record["slide_id"] in test_slide_ids
    ]
    development_slide_ids = sorted(all_supported_slides - test_slide_ids)
    if len(development_slide_ids) < 2:
        raise RuntimeError(
            "The derived holdout left fewer than two development slides."
        )
    development_pool = [
        record for record in combined_supported
        if record["slide_id"] in set(development_slide_ids)
    ]
    WSIBENCH_TEST_SCOPE_LABEL = "derived published-feature holdout"
    WSIBENCH_SPLIT_PROTOCOL = (
        "deterministic slide-disjoint split over feature-supported annotations"
    )
    warnings.warn(
        "Published features do not cover a viable official training split. "
        "Using a deterministic slide-disjoint derived holdout. These results "
        "must not be reported as official WSI-Bench test results."
    )

rng.shuffle(development_slide_ids)
validation_slide_count = min(
    len(development_slide_ids) - 1,
    max(1, int(round(len(development_slide_ids) * WSIBENCH_VALIDATION_FRACTION))),
)
validation_slide_ids = set(development_slide_ids[:validation_slide_count])
wsibench_validation = [
    record for record in development_pool
    if record["slide_id"] in validation_slide_ids
]
wsibench_train = [
    record for record in development_pool
    if record["slide_id"] not in validation_slide_ids
]


def wsibench_debug_limit(records, environment_name):
    limit = int(os.environ.get(environment_name, "0"))
    return records if limit <= 0 else records[:limit]


wsibench_train = wsibench_debug_limit(wsibench_train, "WSIBENCH_MAX_TRAIN")
wsibench_validation = wsibench_debug_limit(
    wsibench_validation, "WSIBENCH_MAX_VALIDATION"
)
wsibench_test = wsibench_debug_limit(wsibench_test, "WSIBENCH_MAX_TEST")

split_summary = pd.DataFrame([
    {
        "split": name,
        "questions": len(records),
        "slides": len({record["slide_id"] for record in records}),
        "open_questions": sum(record["answer_type"] == "Open-ended" for record in records),
        "closed_questions": sum(record["answer_type"] == "Closed-ended" for record in records),
    }
    for name, records in {
        "train": wsibench_train,
        "validation": wsibench_validation,
        WSIBENCH_TEST_SCOPE_LABEL: wsibench_test,
    }.items()
])
display(split_summary)

train_slides = {record["slide_id"] for record in wsibench_train}
validation_slides = {record["slide_id"] for record in wsibench_validation}
assert train_slides.isdisjoint(validation_slides)
assert train_slides.isdisjoint(test_slide_ids)
assert validation_slides.isdisjoint(test_slide_ids)
print("Annotation paths:", {key: str(value) for key, value in annotation_paths.items()})
print("WSI feature root:", WSIBENCH_FEATURE_ROOT)
print("Test mode:", WSIBENCH_TEST_MODE)
print("Feature coverage mode:", WSIBENCH_FEATURE_COVERAGE_MODE)
print("Split protocol:", WSIBENCH_SPLIT_PROTOCOL)

# Prepare WSI feature dataloaders

In [ ]:
from collections import OrderedDict
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

WSIBENCH_MAX_PATCHES = int(os.environ.get("WSIBENCH_MAX_PATCHES", "256"))
WSIBENCH_EXPECTED_FEATURE_DIM = int(
    os.environ.get("WSIBENCH_FEATURE_DIM", "0")
)
WSIBENCH_FEATURE_CACHE_SIZE = int(
    os.environ.get("WSIBENCH_FEATURE_CACHE_SIZE", "32")
)
if WSIBENCH_MAX_PATCHES < 1:
    raise ValueError("WSIBENCH_MAX_PATCHES must be positive.")

WSIBENCH_FEATURE_EXTENSIONS = {".pt", ".pth", ".npy", ".npz", ".h5", ".hdf5"}
WSIBENCH_FEATURE_INDEX = {}
for feature_path in WSIBENCH_FEATURE_ROOT.rglob("*"):
    if feature_path.is_file() and feature_path.suffix.lower() in WSIBENCH_FEATURE_EXTENSIONS:
        WSIBENCH_FEATURE_INDEX.setdefault(feature_path.name, feature_path)
        WSIBENCH_FEATURE_INDEX.setdefault(feature_path.stem, feature_path)
WSIBENCH_INDEXED_FEATURE_COUNT = len(
    {str(path) for path in WSIBENCH_FEATURE_INDEX.values()}
)
print("Indexed WSI feature files:", WSIBENCH_INDEXED_FEATURE_COUNT)


def wsibench_resolve_feature_path(image_reference):
    reference = Path(str(image_reference))
    candidates = [WSIBENCH_FEATURE_ROOT / reference, WSIBENCH_FEATURE_ROOT / reference.name]
    if reference.suffix.lower() not in WSIBENCH_FEATURE_EXTENSIONS:
        candidates.extend(
            WSIBENCH_FEATURE_ROOT / f"{reference.name}{extension}"
            for extension in sorted(WSIBENCH_FEATURE_EXTENSIONS)
        )
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    for key in (reference.name, reference.stem):
        if key in WSIBENCH_FEATURE_INDEX:
            return WSIBENCH_FEATURE_INDEX[key]
    raise FileNotFoundError(
        f"Could not resolve WSI feature '{image_reference}' below {WSIBENCH_FEATURE_ROOT}."
    )


def wsibench_tensor_from_payload(payload):
    if torch.is_tensor(payload):
        return payload
    if isinstance(payload, np.ndarray):
        return torch.from_numpy(payload)
    if isinstance(payload, dict):
        preferred = ("features", "feature", "embeddings", "embedding", "x", "data")
        for key in preferred:
            if key in payload:
                return wsibench_tensor_from_payload(payload[key])
        for value in payload.values():
            try:
                tensor = wsibench_tensor_from_payload(value)
                if tensor.ndim >= 2:
                    return tensor
            except (TypeError, ValueError):
                pass
    if isinstance(payload, (list, tuple)):
        for value in payload:
            try:
                tensor = wsibench_tensor_from_payload(value)
                if tensor.ndim >= 2:
                    return tensor
            except (TypeError, ValueError):
                pass
    raise TypeError("No numeric patch-feature tensor was found in the feature payload.")


def wsibench_load_feature_file(path):
    suffix = path.suffix.lower()
    if suffix in {".pt", ".pth"}:
        try:
            payload = torch.load(path, map_location="cpu", weights_only=True)
        except TypeError:
            payload = torch.load(path, map_location="cpu")
    elif suffix == ".npy":
        payload = np.load(path, allow_pickle=False)
    elif suffix == ".npz":
        archive = np.load(path, allow_pickle=False)
        payload = archive[archive.files[0]]
    elif suffix in {".h5", ".hdf5"}:
        with h5py.File(path, "r") as handle:
            datasets = []
            handle.visititems(
                lambda name, obj: datasets.append(np.asarray(obj))
                if isinstance(obj, h5py.Dataset) and obj.ndim >= 2 else None
            )
        if not datasets:
            raise ValueError(f"No 2-D feature dataset found in {path}")
        payload = datasets[0]
    else:
        raise ValueError(f"Unsupported feature extension: {path.suffix}")

    tensor = wsibench_tensor_from_payload(payload).detach().cpu().float().squeeze()
    if tensor.ndim == 1:
        tensor = tensor.unsqueeze(0)
    elif tensor.ndim > 2:
        tensor = tensor.reshape(-1, tensor.shape[-1])
    if tensor.ndim != 2:
        raise ValueError(f"Expected a 2-D patch-feature tensor in {path}; got {tensor.shape}")

    common_dimensions = {256, 384, 512, 768, 1024, 1280, 1536, 2048, 2560}
    if (
        WSIBENCH_EXPECTED_FEATURE_DIM > 0
        and tensor.shape[0] == WSIBENCH_EXPECTED_FEATURE_DIM
        and tensor.shape[1] != WSIBENCH_EXPECTED_FEATURE_DIM
    ):
        tensor = tensor.transpose(0, 1)
    elif tensor.shape[1] not in common_dimensions and tensor.shape[0] in common_dimensions:
        tensor = tensor.transpose(0, 1)
    tensor = torch.nan_to_num(tensor, nan=0.0, posinf=0.0, neginf=0.0).contiguous()
    if tensor.shape[0] < 1 or tensor.shape[1] < 1:
        raise ValueError(f"Empty feature tensor in {path}")
    return tensor


class WSIBenchDataset(Dataset):
    def __init__(self, records, feature_dim=None, max_patches=WSIBENCH_MAX_PATCHES):
        self.dataset = list(records)
        self.max_patches = int(max_patches)
        self._feature_cache = OrderedDict()
        first_path = wsibench_resolve_feature_path(self.dataset[0]["image"])
        first_tensor = wsibench_load_feature_file(first_path)
        inferred_dim = int(first_tensor.shape[1])
        self.feature_dim = inferred_dim if feature_dim is None else int(feature_dim)
        if inferred_dim != self.feature_dim:
            raise ValueError(
                f"Feature dimension mismatch: expected {self.feature_dim}, got {inferred_dim}"
            )

    def __len__(self):
        return len(self.dataset)

    def _load(self, image_reference):
        path = wsibench_resolve_feature_path(image_reference)
        cache_key = str(path)
        if cache_key in self._feature_cache:
            tensor = self._feature_cache.pop(cache_key)
            self._feature_cache[cache_key] = tensor
            return tensor
        tensor = wsibench_load_feature_file(path)
        if tensor.shape[1] != self.feature_dim:
            raise ValueError(
                f"Inconsistent feature dimension in {path}: "
                f"expected {self.feature_dim}, got {tensor.shape[1]}"
            )
        if WSIBENCH_FEATURE_CACHE_SIZE > 0:
            self._feature_cache[cache_key] = tensor
            while len(self._feature_cache) > WSIBENCH_FEATURE_CACHE_SIZE:
                self._feature_cache.popitem(last=False)
        return tensor

    def __getitem__(self, index):
        sample = self.dataset[index]
        features = self._load(sample["image"])
        # Deterministic uniform token selection. If a slide has fewer tokens,
        # linspace repeats valid tokens rather than introducing padded zeros.
        token_indices = torch.linspace(
            0, features.shape[0] - 1, steps=self.max_patches
        ).round().long()
        features = features[token_indices].clone()
        return features, str(sample["question"]), str(sample["answer"])


# Fail early with a compact missing-file report rather than during an epoch.
required_feature_references = sorted({
    str(record["image"])
    for record in (wsibench_train + wsibench_validation + wsibench_test)
})
missing_feature_references = []
for image_reference in required_feature_references:
    try:
        wsibench_resolve_feature_path(image_reference)
    except FileNotFoundError:
        missing_feature_references.append(image_reference)
        if len(missing_feature_references) >= 20:
            break
if missing_feature_references:
    raise FileNotFoundError(
        "Strict feature coverage failed for the selected annotation splits.\n"
        f"Only {WSIBENCH_INDEXED_FEATURE_COUNT} feature files were indexed below "
        f"{WSIBENCH_FEATURE_ROOT}.\n"
        "First missing annotation references:\n  - "
        + "\n  - ".join(missing_feature_references)
        + "\nComplete the feature collection before rerunning. From the shell, first check "
        + "the download size and then download into the same root:\n"
        + f"  hf download {WSIBENCH_FEATURE_REPO_ID} --repo-type dataset "
        + f"--local-dir \"{WSIBENCH_FEATURE_ROOT}\" --dry-run\n"
        + f"  hf download {WSIBENCH_FEATURE_REPO_ID} --repo-type dataset "
        + f"--local-dir \"{WSIBENCH_FEATURE_ROOT}\"\n"
        + "Do not filter out missing slides for the final evaluation."
    )

train_data = wsibench_train
val_data = wsibench_validation
test_data = wsibench_test

requested_feature_dim = (
    WSIBENCH_EXPECTED_FEATURE_DIM if WSIBENCH_EXPECTED_FEATURE_DIM > 0 else None
)
train_dataset = WSIBenchDataset(train_data, feature_dim=requested_feature_dim)
WSIBENCH_FEATURE_DIM = train_dataset.feature_dim
val_dataset = WSIBenchDataset(val_data, feature_dim=WSIBENCH_FEATURE_DIM)
test_dataset = WSIBenchDataset(test_data, feature_dim=WSIBENCH_FEATURE_DIM)

print({
    "train_questions": len(train_dataset),
    "validation_questions": len(val_dataset),
    "test_questions": len(test_dataset),
    "evaluation_scope": WSIBENCH_TEST_SCOPE_LABEL,
    "split_protocol": WSIBENCH_SPLIT_PROTOCOL,
    "feature_dimension": WSIBENCH_FEATURE_DIM,
    "tokens_per_slide": WSIBENCH_MAX_PATCHES,
})

features, question, answer = train_dataset[0]
plt.figure(figsize=(10, 4))
plt.imshow(features[:64, :128], aspect="auto", cmap="viridis")
plt.colorbar(label="feature value")
plt.xlabel("feature dimension (first 128)")
plt.ylabel("WSI patch token (first 64)")
plt.title(f"Q: {question[:100]}\nA: {answer[:100]}", fontsize=10)
plt.tight_layout()
plt.show()

#Model Architecture

Paper: https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf

GPT-2 uses a decoder-only transformer architecture with multiple model sizes; the commonly used GPT-2 Base model contains 12 transformer blocks (layers), a context window of 1024 tokens, a hidden embedding size of 768, and about 117 million parameters, while larger variants scale up to 48 transformer blocks and 1.5 billion parameters.

[1] Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. OpenAI blog, 1(8), 9.

###Cross-Attention Fusion

In [ ]:
import math
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model

####Cross-Attention Fusion###########
class CrossAttentionFusion(nn.Module):
    def __init__(self, hidden_dim=768, num_heads=8, dropout=0.1):
        super().__init__()

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout)
        )

    def forward(self, text_embeds, image_embeds, text_att_mask=None):
        """
        text_embeds:  [B, T, 768]
        image_embeds: [B, N, 768]
        text attends to image
        """

        attended_text, attn_weights = self.cross_attn(
            query=text_embeds,
            key=image_embeds,
            value=image_embeds,
            need_weights=False
        )

        x = self.norm1(text_embeds + attended_text)
        x = self.norm2(x + self.ffn(x))

        return x

### Gated WSI feature compression

In [ ]:
# The trainable feature gate is implemented in WSIFeatureEncoder below.

### WSI feature encoder + multimodal GPT-2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from peft import get_peft_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class WSIFeatureEncoder(nn.Module):
    """Project and compress precomputed patch features into visual tokens."""

    def __init__(self, input_dim, hidden_dim=768, output_tokens=32):
        super().__init__()
        self.output_tokens = int(output_tokens)
        self.projection = nn.Linear(int(input_dim), hidden_dim, bias=False)
        self.normalization = nn.LayerNorm(hidden_dim)
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Linear(hidden_dim // 4, 1),
            nn.Sigmoid(),
        )

    def forward(self, features):
        if features.ndim != 3:
            raise ValueError(
                f"Expected WSI features [batch, patches, dimension]; got {features.shape}"
            )
        tokens = self.normalization(self.projection(features))
        tokens = tokens * self.gate(tokens)
        tokens = F.adaptive_avg_pool1d(
            tokens.transpose(1, 2), self.output_tokens
        ).transpose(1, 2)
        return tokens


class MedVQA(nn.Module):
    def __init__(self, peft_config=None):
        super().__init__()
        output_tokens = int(os.environ.get("WSIBENCH_VISUAL_TOKENS", "32"))
        self.visual_encoder = WSIFeatureEncoder(
            input_dim=WSIBENCH_FEATURE_DIM,
            hidden_dim=768,
            output_tokens=output_tokens,
        )

        self.tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        self.tokenizer.pad_token = self.tokenizer.eos_token

        gpt = GPT2LMHeadModel.from_pretrained("gpt2")
        self.gpt = get_peft_model(gpt, peft_config)
        self.fusion = CrossAttentionFusion(hidden_dim=768, num_heads=4)

    def forward(self, image, qa_inputs_ids, qa_att_mask):
        image_embeds = self.visual_encoder(image)
        text_embeds = self.gpt.get_input_embeddings()(qa_inputs_ids)
        fused_embeds = self.fusion(
            text_embeds=text_embeds,
            image_embeds=image_embeds,
            text_att_mask=qa_att_mask,
        )
        return self.gpt(
            inputs_embeds=fused_embeds,
            attention_mask=qa_att_mask,
        ).logits

# Model training or checkpoint reuse

This notebook reuses its own WSI-Bench checkpoint by default:

`checkpoints_wsibench_pa_she/best_model_ca_wsibench_pa_she.pth`

If the file is absent, training runs for up to **10 epochs** with validation
early stopping (patience **5**) and saves the best model. Set
`REUSE_TRAINED_CHECKPOINT=0` to deliberately retrain. This checkpoint is
separate from the PathVQA and VQA-RAD checkpoints because the visual encoder
and feature dimension are WSI-Bench-specific.

In [ ]:
#Training Script for Multimodal GPT2 with LoRA
import os
import torch
import argparse
import torch.utils.data
import numpy as np
import random

from torch import nn
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer

import evaluate
from nltk.translate.bleu_score import corpus_bleu
from peft import  TaskType, LoraConfig

import warnings
warnings.filterwarnings('ignore')

REUSE_TRAINED_CHECKPOINT = os.environ.get(
    'REUSE_TRAINED_CHECKPOINT', '1'
).strip().lower() not in {'0', 'false', 'no'}


def load_vqa_checkpoint(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def adjust_learning_rate(optimizer, shrink_factor):
    print("\nDECAYING learning rate.")
    for param_group in optimizer.param_groups:
        param_group['lr'] = param_group['lr'] * shrink_factor
    print("The new learning rate is %f\n" % (optimizer.param_groups[0]['lr'],))

def train(args, train_dataloader, model, criterion, optimizer, epoch, tokenizer, device):
    model.train()
    total_loss = []

    for i, (images, questions, answers) in enumerate(train_dataloader, 0):
        # prepare prompts
        qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
        qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

        # get labels
        labels = qa_prompt_inputs['input_ids'].clone()
        labels = labels.to(device)

        # for labels, mask question tokens and padding tokens
        for idx, q in enumerate(questions):
            q_prompt = f"Question: {q}\nAnswer: "
            q_length = len(tokenizer(q_prompt)["input_ids"]) - 1

            labels[idx, :q_length] = -100  # mask question
            eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
            if eos_mask.sum() > 1:  # if more than 1 EOS
                first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS

        # get logits and labels
        logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
        )

        # get shifted logits and labels
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        # compute loss
        shift_logits = shift_logits.view(-1, shift_logits.size(-1))
        shift_labels = shift_labels.view(-1)
        loss = criterion(shift_logits, shift_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss.append(loss.item())
        if i%50 == 0:
            print("Training - Epoch: {}/{}, Iteration: {}/{}, Training Loss: {:.6f}".format(epoch, args.epochs, i, len(train_dataloader), np.array(total_loss).mean()))


def validate(args, val_loader, model, criterion, epoch, tokenizer, device):
    total_loss = []
    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(val_loader, 0):
            # prepare prompts
            qa_prompt = [f'Question: {q}\nAnswer: {a}' for q, a in zip(questions, answers)]
            qa_prompt_inputs = tokenizer(qa_prompt, truncation=True, padding="max_length", max_length=int(args.seq_length), return_tensors="pt")

            # get labels
            labels = qa_prompt_inputs['input_ids'].clone()
            labels = labels.to(device)

            # for labels, mask question tokens and padding tokens
            answer_starts = []
            answer_ends = []
            for idx, q in enumerate(questions):
                q_prompt = f"Question: {q}\nAnswer: "
                q_length = len(tokenizer(q_prompt)["input_ids"]) - 1
                answer_starts.append(q_length+1)

                labels[idx, :q_length] = -100  # mask question
                eos_mask = (labels[idx] == tokenizer.eos_token_id)  # get all EOS position
                if eos_mask.sum() > 1:  # if more than 1 EOS
                    first_eos_pos = eos_mask.nonzero()[0].item()  # get first EOS position
                    labels[idx, (first_eos_pos+1):] = -100  # mask paddings, left one EOS
                    answer_ends.append(first_eos_pos)

            # get logits and labels
            logits = model(
                image=images.to(device),
                qa_inputs_ids=qa_prompt_inputs['input_ids'].to(device),
                qa_att_mask=qa_prompt_inputs['attention_mask'].to(device)
            )

            # get shifted logits and labels
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            # compute loss
            shift_logits = shift_logits.view(-1, shift_logits.size(-1))
            shift_labels = shift_labels.view(-1)
            loss = criterion(shift_logits, shift_labels)
            total_loss.append(loss.item())

    return np.array(total_loss).mean()


def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)


def get_arg():
    parser = argparse.ArgumentParser(description='VisualQuestionAnswerGeneration')
    # Training parameters
    parser.add_argument('--epochs',         type=int,   default=10,   help='number of epochs to train for')
    parser.add_argument('--batch_size',     type=int,   default=32,   help='training and validation batch size')
    parser.add_argument('--workers',        type=int,   default=8,    help='for data-loading')
    parser.add_argument('--random_seed',    type=int,   default=42,   help='random seed')
    parser.add_argument('--seq_length',     type=int,   default=160,  help='sequence length for question and answer')
    parser.add_argument('--dropout', type=float, default=0.1, help='dropout')
    parser.add_argument('--early_stopping_patience', type=int, default=5,
                        help='stop after this many epochs without validation improvement')

    parser.add_argument('--dataset',        default='endo',  help='endo / pit')
    parser.add_argument('--lr',             type=float, default=0.0002,  help='0.0000001, 0.00000005')
    parser.add_argument('--checkpoint_dir', default='checkpoints_wsibench_pa_she/',
                        help='separate checkpoint path for the PA-SHE experiment')

    args = parser.parse_args([])
    return args


if __name__ == '__main__':

    args = get_arg()
    seed_everything(args.random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f'Batch size: {args.batch_size}')
    print(f'Learning rate: {args.lr}')
    print(f'Random seed: {args.random_seed}')
    print(f'Sequence length: {args.seq_length}')
    print(f'Maximum epochs: {args.epochs}')
    print(f'Early-stopping patience: {args.early_stopping_patience}')

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    MODEL_CHECKPOINT_PATH = os.path.join(
        args.checkpoint_dir,
        'best_model_ca_wsibench_pa_she.pth',
    )
    reuse_checkpoint = (
        REUSE_TRAINED_CHECKPOINT
        and os.path.isfile(MODEL_CHECKPOINT_PATH)
    )
    checkpoint_saved_this_run = False
    start_epoch = 1
    epochs_since_improvement = 0
    best_val_loss = float('inf')

    print('Dataset: WSI-Bench slide-disjoint train/validation splits')
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=args.workers > 0,
    )

    print(
        'DataLoader configuration:',
        {
            'train_examples': len(train_dataset),
            'validation_examples': len(val_dataset),
            'batch_size': args.batch_size,
            'workers': args.workers,
        },
    )

    # init tokenizer and model
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    model = model.to(device)

    # for name, param in model.named_parameters():
    #     if param.requires_grad:
    #         print(name)

    pytorch_total_params = sum(p.numel() for p in model.parameters())
    print('model params: ', pytorch_total_params)

    # init optimizer and criterion
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=-100).to(device)

    # Reuse this notebook's validation-selected checkpoint unless retraining
    # is explicitly requested or the checkpoint does not exist.
    if reuse_checkpoint:
        print(
            'Reusing trained checkpoint; skipping epoch training:',
            MODEL_CHECKPOINT_PATH,
        )
        training_epochs = []
    else:
        if REUSE_TRAINED_CHECKPOINT:
            print(
                'No existing checkpoint found; training from scratch:',
                MODEL_CHECKPOINT_PATH,
            )
        else:
            print('Checkpoint reuse disabled; training from scratch.')
        print('Start training.')
        training_epochs = range(start_epoch, args.epochs + 1)

    for epoch in training_epochs:
        if epochs_since_improvement > 0 and epochs_since_improvement % 5 == 0:
            adjust_learning_rate(optimizer, 0.8)

        # train
        train(args, train_dataloader=train_dataloader, model=model, criterion=criterion, optimizer=optimizer,
              epoch=epoch, tokenizer=tokenizer, device=device)
        # validation
        val_loss = validate(args, val_loader=val_dataloader, model=model, criterion=criterion,
                            epoch=epoch, tokenizer=tokenizer, device=device)

        if val_loss < best_val_loss:  # save model with better validation loss
            epochs_since_improvement = 0
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODEL_CHECKPOINT_PATH)
            checkpoint_saved_this_run = True
            model.tokenizer.save_pretrained(args.checkpoint_dir)
            print('Best validation loss, model saved.')
        else:
            epochs_since_improvement += 1
            print("\nEpochs since last improvement: %d\n" % (epochs_since_improvement,))

        if epochs_since_improvement >= args.early_stopping_patience:
            print(
                f'Early stopping at epoch {epoch}: validation loss did not improve '
                f'for {args.early_stopping_patience} consecutive epochs.'
            )
            break

    if not reuse_checkpoint:
        if not checkpoint_saved_this_run:
            raise RuntimeError(
                'Training finished without producing a validation checkpoint.'
            )
        print(f'End training. Best validation loss: {best_val_loss:.6f}')

    if not os.path.isfile(MODEL_CHECKPOINT_PATH):
        raise FileNotFoundError(
            f'Model checkpoint not found: {MODEL_CHECKPOINT_PATH}'
        )
    model.load_state_dict(
        load_vqa_checkpoint(MODEL_CHECKPOINT_PATH, device)
    )
    model.to(device)
    model.eval()
    print('Loaded validation-selected checkpoint:', MODEL_CHECKPOINT_PATH)

# Validation inference: qualitative WSI-feature sanity check

In [ ]:
# Validation-only prediction visualisations (selected test scope remains untouched)
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F


def greedy_search_single(image, question, model, tokenizer, max_length, device):
    model.eval()
    with torch.no_grad():
        prompt = tokenizer(
            f"Question: {question}\nAnswer:",
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = prompt["input_ids"].to(device)
        attention = prompt["attention_mask"].to(device)
        generated = []
        image = image.unsqueeze(0).to(device)
        for _ in range(max(0, max_length - input_ids.shape[1])):
            logits = model(
                image=image,
                qa_inputs_ids=input_ids,
                qa_att_mask=attention,
            )[0, -1]
            next_id = torch.argmax(logits)
            if int(next_id) == tokenizer.eos_token_id:
                break
            generated.append(int(next_id))
            input_ids = torch.cat([input_ids, next_id.view(1, 1)], dim=1)
            attention = torch.cat([
                attention,
                torch.ones((1, 1), dtype=attention.dtype, device=device),
            ], dim=1)
        return tokenizer.decode(generated, skip_special_tokens=True).strip()


def inference_few_samples(sample_indices=(2, 7, 11)):
    valid_indices = [index for index in sample_indices if index < len(val_dataset)]
    if not valid_indices:
        raise ValueError("No requested validation index exists.")
    fig, axes = plt.subplots(len(valid_indices), 1, figsize=(13, 4 * len(valid_indices)))
    axes = np.atleast_1d(axes)
    for axis, index in zip(axes, valid_indices):
        features, question, reference = val_dataset[index]
        prediction = greedy_search_single(
            features, question, model, tokenizer, max_length=args.seq_length, device=device
        )
        axis.imshow(features[:64, :128], aspect="auto", cmap="viridis")
        axis.set_title(
            f"Q: {question[:160]}\nReference: {reference[:160]}\nPrediction: {prediction[:160]}",
            fontsize=9,
        )
        axis.set_xlabel("feature dimension")
        axis.set_ylabel("WSI patch token")
    plt.tight_layout()
    plt.show()


inference_few_samples()

# Validation inference: utility sanity check

This is not the final reported test utility. Official-test utility is computed
later from the greedy predictions generated after the clustering lock.

In [ ]:
# Validation-only utility sanity metrics (test remains untouched)
import os
import re
import numpy as np
import random

import torch
import torch.utils.data
from torch import nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms.functional import InterpolationMode
import torch.nn as nn
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ViTModel, BlipTextModel
from peft import get_peft_model
from peft import  TaskType, LoraConfig

from PIL import Image
from tqdm import tqdm
import evaluate
rouge = evaluate.load("rouge")
import time
import math
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


def batch_greedy_search(images, questions, model, tokenizer, max_length, device):
    answers = []
    batch_size = len(questions)

    model.eval()
    with torch.no_grad():
        # Prepare the prompts for the entire batch
        prompt_texts = [f"Question: {q}\nAnswer:" for q in questions]

        # Tokenize the prompts with padding to handle varying lengths
        prompt_inputs = tokenizer(
            prompt_texts,
            return_tensors="pt",
            padding='longest',
            truncation=True,
            max_length=max_length - 1,
            add_special_tokens=False
        )

        # Prepare model inputs
        padded_input_ids = torch.zeros((batch_size, max_length), dtype=torch.long, device=device)
        padded_attention_mask = torch.zeros((batch_size, max_length), device=device)

        orig_length = prompt_inputs['input_ids'].size(1)
        padded_input_ids[:, :orig_length] = prompt_inputs['input_ids'].to(device)
        padded_attention_mask[:, :orig_length] = prompt_inputs['attention_mask'].to(device)

        images = images.to(device)

        # Initialize tensors to store generated tokens
        only_answer_ids = torch.empty((batch_size, 0), dtype=torch.long, device=device)

        # Track which sequences have finished generating
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Record each sample length (number of non-eos tokens)
        valid_lengths = padded_attention_mask.sum(dim=1).long()
        batch_indices = torch.arange(batch_size, device=device)

        for _ in range(max_length - orig_length):
            max_valid_lengths = valid_lengths.max().item()

            logits = model(
                image=images,
                qa_inputs_ids=padded_input_ids[:, :max_valid_lengths],
                qa_att_mask=padded_attention_mask[:, :max_valid_lengths]
            )

            last_valid_logits = logits[batch_indices, valid_lengths - 1, :]
            next_token_ids = torch.argmax(last_valid_logits, dim=-1)

            is_eos = next_token_ids == tokenizer.eos_token_id
            finished = finished | is_eos

            padded_input_ids[batch_indices, valid_lengths] = next_token_ids
            padded_attention_mask[batch_indices, valid_lengths] = 1
            valid_lengths += 1

            only_answer_ids = torch.cat(
                [only_answer_ids, next_token_ids.unsqueeze(1)],
                dim=1
            )

            if finished.all():
                break

        # Decode the generated tokens into strings
        generated_ids_cpu = only_answer_ids.cpu().tolist()  # Move to CPU and convert to list for processing
        for i in range(batch_size):
            # Find the first occurrence of eos_token_id to truncate the answer
            try:
                eos_index = generated_ids_cpu[i].index(tokenizer.eos_token_id)
                answer_ids = generated_ids_cpu[i][:eos_index]
            except ValueError:
                # If eos_token_id is not found, use all generated tokens
                answer_ids = generated_ids_cpu[i]

            # Decode the token IDs to a string, skipping special tokens
            answer = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
            answers.append(answer)

    return answers

def evaluate_vqa_split(args, data_loader, model, tokenizer, device):
    references = []
    hypotheses = []

    model.eval()
    with torch.no_grad():
        for i, (images, questions, answers) in enumerate(tqdm(data_loader), 0):
            images = images.to(device)
            generated_answers = batch_greedy_search(
                images,
                questions,
                model,
                tokenizer,
                max_length=args.seq_length,
                device=device
            )

            references.extend(answers)
            hypotheses.extend(generated_answers)

    return references, hypotheses

def normalize_vqa_metric_text(text):
    text = re.sub(r"[^a-z0-9%.\-\s]", " ", str(text).lower().strip())
    return re.sub(r"\s+", " ", text).strip() or "<empty>"


def get_nlp_mettics(references, hypotheses):
    references = [normalize_vqa_metric_text(text) for text in references]
    hypotheses = [normalize_vqa_metric_text(text) for text in hypotheses]
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    meteor = evaluate.load('meteor')

    # compute HF metrics
    results_bleu = bleu.compute(
        predictions=hypotheses, references=references, max_order=1
    )
    results_rouge = rouge.compute(predictions=hypotheses, references=references)
    results_meteor = meteor.compute(predictions=hypotheses, references=references)

    print("HuggingFace Metrics Results:")

    print(f"BLEU-1: {results_bleu['bleu']:.6f}")
    print(f"RougeL: {results_rouge['rougeL']:.6f}")
    print(f"Meteor: {results_meteor['meteor']:.6f}")


if __name__ == '__main__':
    # parameters
    random_seed = 42
    seed_everything(random_seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_dataset = val_dataset
    validation_dataloader = DataLoader(validation_dataset, batch_size=64, shuffle=False, num_workers=4)
    print('Full validation size (test remains untouched):', len(validation_dataset))

    # load weights
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"]
    )

    model = MedVQA(peft_config=lora_config)
    save_dir = MODEL_CHECKPOINT_PATH
    # save_dir = f'best_model_ca_lr1.pth'
    model.load_state_dict(load_vqa_checkpoint(save_dir, device))
    model.to(device)
    model.eval()

    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    tokenizer.pad_token = tokenizer.eos_token

    validation_references, validation_hypotheses = evaluate_vqa_split(
        args,
        data_loader=validation_dataloader,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    get_nlp_mettics(validation_references, validation_hypotheses)

# Perturbation-Aware Semantic Hallucination Entropy (PA-SHE) for WSI-Bench

## Experimental flow

```mermaid
flowchart TB
    A["Feature-supported WSI-Bench annotations"] --> B["Slide-disjoint train and validation split"]
    B --> C["Train or reuse WSI feature + GPT-2 LoRA model"]
    C --> D["Frozen validation sampling under four conditions"]
    D --> E1["Exact-text clustering"]
    D --> E2["SBERT and BGE cosine grids<br/>0.70, 0.80, 0.90"]
    D --> E3["RoBERTa/DeBERTa mutual-NLI grids<br/>0.35, 0.50, 0.65"]
    E1 --> F["Validation grid over clustering,<br/>length alpha and weight temperature"]
    E2 --> F
    E3 --> F
    F --> G["Lock predicted-closed and<br/>predicted-open configurations"]
    H["Held-out open + closed questions"] --> I["Frozen test sampling under four conditions"]
    G --> J["Question-text routing + validation percentile calibration"]
    I --> J
    J --> K["Overall, closed-ended and open-ended safety"]
    J --> N["Raw weighting ablation<br/>alpha=0, T=1"]
    K --> L["Primary label: ROUGE-L below 0.50"]
    K --> M["Sensitivity only: 0.30 and 0.70"]
```

## 1. Configuration and reproducibility

The complete slide-disjoint validation subset is used for clustering and
sequence-weight calibration selection. The held-out test annotations are used
once for locked evaluation when
`PASHE_MAX_EXAMPLES = None`. Sample and feature caches are tied to the checkpoint,
dataset contents, generation settings, split, perturbation version, clustering,
length alpha, and weight temperature.

For a shorter runtime, PA-SHE answers are generated in condition-level batches.
The retained comparison is limited to SE, SNNE, embedding QA-SNNE, VASE,
raw-weight PA-SHE, and calibrated PA-SHE.

In [ ]:
# Install once if needed:
# !pip install -q pandas scipy scikit-learn sentence-transformers transformers rouge-score seaborn

import gc
import os
import hashlib
import json
import math
from pathlib import Path
import random
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from rouge_score import rouge_scorer
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import average_precision_score, roc_auc_score
from tqdm.auto import tqdm

_pashe_max_examples = int(os.environ.get("PASHE_MAX_EXAMPLES", "0"))
PASHE_MAX_EXAMPLES = None if _pashe_max_examples <= 0 else _pashe_max_examples
PASHE_NUM_SAMPLES = 10
PASHE_GENERATION_BATCH_SIZE = int(os.environ.get("PASHE_GENERATION_BATCH_SIZE", "20"))
PASHE_MAX_NEW_TOKENS = 48
PASHE_TEMPERATURE = 1.0
PASHE_TOP_P = 0.90
PASHE_RANDOM_SEED = 42
PASHE_PERTURBATION_VERSION = 3

# Question-Aligned Semantic Nearest Neighbor Entropy (QA-SNNE).
QA_SNNE_NUM_SAMPLES = int(os.environ.get("QA_SNNE_NUM_SAMPLES", "20"))
QA_SNNE_TEMPERATURE = 1.0
QA_SNNE_TOP_K = 50
QA_SNNE_TOP_P = 0.90
QA_SNNE_BETA = 10.0
QA_SNNE_TAU = 1.0
QA_SNNE_CACHE_SCHEMA_VERSION = 2
QA_SNNE_EMBEDDING_MODEL = "pritamdeka/S-PubMedBert-MS-MARCO"
QA_SNNE_VARIANTS = {
    "Embedding": "qa_snne_embedding",
}
if QA_SNNE_NUM_SAMPLES < 2:
    raise ValueError("QA_SNNE_NUM_SAMPLES must be at least two.")
if PASHE_GENERATION_BATCH_SIZE < 1:
    raise ValueError("PASHE_GENERATION_BATCH_SIZE must be positive.")

PASHE_LABEL_THRESHOLDS = [0.30, 0.50, 0.70]
PASHE_PRIMARY_LABEL_THRESHOLD = 0.50

# Predeclared method/threshold grid. Validation selects; test never does.
PASHE_SBERT_THRESHOLDS = [0.70, 0.80, 0.90]
PASHE_BGE_THRESHOLDS = [0.70, 0.80, 0.90]
PASHE_ROBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
PASHE_DEBERTA_NLI_THRESHOLDS = [0.35, 0.50, 0.65]
LENGTH_ALPHA_GRID = [0.0, 0.5, 1.0]
WEIGHT_TEMPERATURE_GRID = [0.5, 1.0, 2.0, 4.0]
PASHE_WEIGHTING_CANDIDATES = [
    (length_alpha, weight_temperature)
    for length_alpha in LENGTH_ALPHA_GRID
    for weight_temperature in WEIGHT_TEMPERATURE_GRID
]
PASHE_CONTROL_CLUSTERING = "Exact text"
PASHE_SELECTION_CANDIDATES = (
    [PASHE_CONTROL_CLUSTERING]
    + [f"SBERT@{threshold:.2f}" for threshold in PASHE_SBERT_THRESHOLDS]
    + [f"BGE@{threshold:.2f}" for threshold in PASHE_BGE_THRESHOLDS]
    + [
        f"RoBERTa-NLI@{threshold:.2f}"
        for threshold in PASHE_ROBERTA_NLI_THRESHOLDS
    ]
    + [
        f"DeBERTa-NLI@{threshold:.2f}"
        for threshold in PASHE_DEBERTA_NLI_THRESHOLDS
    ]
)

PASHE_SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
PASHE_BGE_MODEL = "BAAI/bge-small-en-v1.5"
PASHE_ROBERTA_NLI_MODEL = "roberta-large-mnli"
PASHE_DEBERTA_NLI_MODEL = "microsoft/deberta-large-mnli"
PASHE_MODEL_BATCH_SIZE = 64
PASHE_CACHE_SCHEMA_VERSION = 5
PASHE_FEATURE_SCHEMA_VERSION = 8

required = [
    "model", "tokenizer", "device", "train_dataset", "val_dataset",
    "test_dataset", "MODEL_CHECKPOINT_PATH",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the original notebook through model loading/evaluation first. Missing: "
        + ", ".join(missing)
    )

PASHE_CHECKPOINT_PATH = Path(MODEL_CHECKPOINT_PATH).resolve()
if not PASHE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {PASHE_CHECKPOINT_PATH}")
PASHE_CACHE_DIR = (
    PASHE_CHECKPOINT_PATH.parent / "pa_she_cache_wsibench"
)
PASHE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

random.seed(PASHE_RANDOM_SEED)
np.random.seed(PASHE_RANDOM_SEED)
torch.manual_seed(PASHE_RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(PASHE_RANDOM_SEED)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

print({
    "train_examples": len(train_dataset),
    "validation_examples": len(val_dataset),
    "test_examples": len(test_dataset),
    "samples_per_condition": PASHE_NUM_SAMPLES,
    "selection_split": "validation",
    "test_used_for_selection": False,
    "selection_label_threshold": PASHE_PRIMARY_LABEL_THRESHOLD,
    "selection_candidates": PASHE_SELECTION_CANDIDATES,
    "length_alpha_grid": LENGTH_ALPHA_GRID,
    "weight_temperature_grid": WEIGHT_TEMPERATURE_GRID,
    "test_label_sensitivity": PASHE_LABEL_THRESHOLDS,
    "cache_directory": str(PASHE_CACHE_DIR),
})

## 2. WSI-feature perturbations and sampled generation

The frozen VQA model receives four conditions for the same slide/question:

- **original:** unmodified WSI patch features and original question;
- **weak:** low-amplitude feature noise representing a mild visual change;
- **distorted:** stronger feature noise plus patch-token replacement, representing
  partial loss/corruption of WSI evidence;
- **paraphrase:** original features with a meaning-preserving question rewrite.

These are embedding-space perturbations because WSI-Bench annotations reference
precomputed WSI patch features rather than ordinary RGB images.

In [ ]:
def pashe_normalize(text):
    return normalize_vqa_metric_text(text)


PASHE_QUESTION_ROUTER_VERSION = 2
PASHE_CLOSED_QUESTION_PREFIXES = (
    "is " , "are " , "was " , "were " , "do " , "does " ,
    "did " , "can " , "could " , "will " , "would " ,
    "has " , "have " , "had " , "should " , "may " , "might " ,
)


def pashe_predict_question_type(question):
    normalized = re.sub(r"\s+", " ", str(question).lower().strip())
    has_options = bool(re.search(r"(?:^|\s)[a-d][\).:]\s", normalized))
    return (
        "Closed"
        if normalized.startswith(PASHE_CLOSED_QUESTION_PREFIXES) or has_options
        else "Open"
    )


PASHE_NEGATION_TERMS = {"no", "not", "without", "absent"}
PASHE_LATERALITY_TERMS = {"left", "right", "bilateral"}


def pashe_preserved_terms(text, vocabulary):
    tokens = set(re.findall(r"[a-z0-9]+", str(text).lower()))
    return tokens.intersection(vocabulary)


def pashe_paraphrase_constraints_hold(original, candidate):
    if not str(candidate).strip():
        return False
    if pashe_predict_question_type(original) != pashe_predict_question_type(candidate):
        return False
    original_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(original))
    candidate_numbers = re.findall(r"\b\d+(?:\.\d+)?\b", str(candidate))
    if original_numbers != candidate_numbers:
        return False
    for protected in (PASHE_NEGATION_TERMS, PASHE_LATERALITY_TERMS):
        if pashe_preserved_terms(original, protected) != pashe_preserved_terms(candidate, protected):
            return False
    return pashe_normalize(original) != pashe_normalize(candidate)


def pashe_paraphrase(question):
    q = re.sub(r"\s+", " ", str(question).strip()).rstrip("?")
    if not q:
        return str(question)
    rules = [
        (r"^what does (?:this|the) (?:image|slide) show$", "What is shown in the slide?"),
        (r"^what is shown in (?:this|the) (?:image|slide)$", "What does the slide show?"),
        (r"^is there (.+)$", r"Does the image show \1?"),
        (r"^does (?:this|the) (?:image|slide) show (.+)$", r"Is \1 visible in the slide?"),
        (r"^is (.+) present$", r"Does the slide show \1?"),
        (r"^are there (.+)$", r"Does the slide contain \1?"),
        (r"^where is (.+) located$", r"What is the location of \1?"),
        (r"^how many (.+) are (?:there|present)$", r"What number of \1 are present?"),
        (r"^what is (?:the )?diagnosis$", "Which diagnosis is most consistent with the slide?"),
        (r"^what is present$", "What finding is present?"),
    ]
    for pattern, replacement in rules:
        match = re.fullmatch(pattern, q, flags=re.IGNORECASE)
        if match:
            candidate = match.expand(replacement).strip()
            if pashe_paraphrase_constraints_hold(q, candidate):
                return candidate
    first_word = q.split(maxsplit=1)[0].lower()
    open_body = q[0].lower() + q[1:] if first_word in {
        "what", "where", "when", "why", "who", "which", "how"
    } else q
    fallback = (
        f"{q} according to the whole-slide image?"
        if pashe_predict_question_type(q) == "Closed"
        else f"Based on the whole-slide image, {open_body}?"
    )
    return fallback if pashe_paraphrase_constraints_hold(q, fallback) else f"{q}?"


def pashe_feature_scale(features):
    scale = features.std(dim=0, keepdim=True, unbiased=False)
    return scale.clamp_min(1e-6)


def pashe_feature_generator(seed):
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    return generator


def pashe_weak_image(image, seed):
    x = image.detach().cpu().float().clone()
    noise = torch.randn(x.shape, generator=pashe_feature_generator(seed))
    noise = 0.015 * pashe_feature_scale(x) * noise
    return x + noise


def pashe_distorted_image(image, seed):
    x = image.detach().cpu().float().clone()
    scale = pashe_feature_scale(x)
    generator = pashe_feature_generator(seed)
    x = x + 0.08 * scale * torch.randn(x.shape, generator=generator)
    replacement_count = max(1, int(round(0.20 * x.shape[0])))
    replacement_indices = torch.randperm(
        x.shape[0], generator=generator
    )[:replacement_count]
    replacement = x.mean(dim=0, keepdim=True)
    x[replacement_indices] = replacement
    return x


def pashe_top_p_filter(logits, top_p):
    if top_p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
    cumulative = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    remove = cumulative > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits = sorted_logits.masked_fill(remove, float("-inf"))
    filtered = torch.full_like(logits, float("-inf"))
    return filtered.scatter(-1, sorted_indices, sorted_logits)


@torch.inference_mode()
def pashe_generate_batch(images, question, do_sample=True):
    """Generate one answer per WSI feature tensor in a single model batch."""
    if not images:
        return []
    prompt = f"Question: {question}\nAnswer:"
    prompt_limit = max(8, int(args.seq_length) - PASHE_MAX_NEW_TOKENS)
    encoded = tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False,
        truncation=True, max_length=prompt_limit,
    )
    batch_size = len(images)
    input_ids = encoded["input_ids"].to(device).repeat(batch_size, 1)
    attention = encoded["attention_mask"].to(device).repeat(batch_size, 1)
    image_batch = torch.stack([image.detach().cpu().float() for image in images]).to(device)
    generated_ids = [[] for _ in range(batch_size)]
    sequence_logprobs = [[] for _ in range(batch_size)]
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for _ in range(PASHE_MAX_NEW_TOKENS):
        scaled = model(
            image=image_batch, qa_inputs_ids=input_ids, qa_att_mask=attention,
        )[:, -1, :] / PASHE_TEMPERATURE
        log_probs = torch.log_softmax(scaled, dim=-1)
        if do_sample:
            sampling_logits = pashe_top_p_filter(scaled, PASHE_TOP_P)
            next_ids = torch.distributions.Categorical(logits=sampling_logits).sample()
        else:
            next_ids = torch.argmax(scaled, dim=-1)
        next_ids = torch.where(
            finished, torch.full_like(next_ids, tokenizer.eos_token_id), next_ids
        )
        selected_logps = log_probs.gather(1, next_ids[:, None]).squeeze(1)
        for sample_index, token_id in enumerate(next_ids.detach().cpu().tolist()):
            if not finished[sample_index] and token_id != tokenizer.eos_token_id:
                generated_ids[sample_index].append(token_id)
                sequence_logprobs[sample_index].append(
                    float(selected_logps[sample_index].item())
                )
        finished = finished | (next_ids == tokenizer.eos_token_id)
        input_ids = torch.cat([input_ids, next_ids[:, None]], dim=1)
        attention = torch.cat([
            attention,
            torch.ones((batch_size, 1), dtype=attention.dtype, device=device),
        ], dim=1)
        if bool(finished.all()):
            break

    return [{
        "answer": tokenizer.decode(token_ids, skip_special_tokens=True).strip(),
        "sequence_logprob": float(np.sum(logps)) if logps else -50.0,
    } for token_ids, logps in zip(generated_ids, sequence_logprobs)]


def pashe_generate_one(image, question, do_sample=True):
    return pashe_generate_batch([image], question, do_sample=do_sample)[0]


@torch.inference_mode()
def qa_snne_generate_samples(image, question, num_samples):
    """Generate QA-SNNE samples from the original input in one batch."""
    prompt = f"Question: {question}\nAnswer:"
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to(device).repeat(num_samples, 1)
    attention = encoded["attention_mask"].to(device).repeat(num_samples, 1)
    image_batch = image.unsqueeze(0).to(device).repeat(num_samples, 1, 1)
    generated_ids = [[] for _ in range(num_samples)]
    finished = torch.zeros(num_samples, dtype=torch.bool, device=device)
    for _ in range(PASHE_MAX_NEW_TOKENS):
        next_logits = model(
            image=image_batch, qa_inputs_ids=input_ids, qa_att_mask=attention
        )[:, -1, :] / QA_SNNE_TEMPERATURE
        top_k = min(QA_SNNE_TOP_K, next_logits.shape[-1])
        if top_k > 0:
            kth = torch.topk(next_logits, top_k, dim=-1).values[:, -1:]
            next_logits = next_logits.masked_fill(
                next_logits < kth, float("-inf")
            )
        next_logits = pashe_top_p_filter(next_logits, QA_SNNE_TOP_P)
        next_ids = torch.distributions.Categorical(logits=next_logits).sample()
        next_ids = torch.where(
            finished, torch.full_like(next_ids, tokenizer.eos_token_id), next_ids
        )
        for sample_index, token_id in enumerate(next_ids.detach().cpu().tolist()):
            if not finished[sample_index] and token_id != tokenizer.eos_token_id:
                generated_ids[sample_index].append(token_id)
        finished = finished | (next_ids == tokenizer.eos_token_id)
        input_ids = torch.cat([input_ids, next_ids[:, None]], dim=1)
        attention = torch.cat([
            attention,
            torch.ones((num_samples, 1), dtype=attention.dtype, device=device),
        ], dim=1)
        if bool(finished.all()):
            break
    return [
        tokenizer.decode(token_ids, skip_special_tokens=True).strip()
        for token_ids in generated_ids
    ]


def pashe_collect_example(
    dataset,
    dataset_index,
    split_name,
):
    image, question, reference = dataset[dataset_index]
    paraphrase = pashe_paraphrase(question)
    split_offset = 0 if str(split_name).lower().startswith("val") else 10_000_000
    perturbation_seed = PASHE_RANDOM_SEED + split_offset + int(dataset_index) * 1000
    condition_inputs = {
        "original": ([image.clone() for _ in range(PASHE_NUM_SAMPLES)], question),
        "weak": ([
            pashe_weak_image(image, perturbation_seed + 100 + sample_index)
            for sample_index in range(PASHE_NUM_SAMPLES)
        ], question),
        "distorted": ([
            pashe_distorted_image(image, perturbation_seed + 200 + sample_index)
            for sample_index in range(PASHE_NUM_SAMPLES)
        ], question),
        "paraphrase": ([image.clone() for _ in range(PASHE_NUM_SAMPLES)], paraphrase),
    }
    records = []
    for condition, (condition_images, condition_question) in condition_inputs.items():
        for start in range(0, len(condition_images), PASHE_GENERATION_BATCH_SIZE):
            batch_records = pashe_generate_batch(
                condition_images[start:start + PASHE_GENERATION_BATCH_SIZE],
                condition_question,
                do_sample=True,
            )
            for record in batch_records:
                record["condition"] = condition
                records.append(record)
    greedy = pashe_generate_one(
        image,
        question,
        do_sample=False,
    )["answer"]
    return {
        "cache_schema_version": PASHE_CACHE_SCHEMA_VERSION,
        "split": str(split_name),
        "dataset_index": int(dataset_index),
        "question": question,
        "paraphrase": paraphrase,
        "paraphrase_changed": pashe_normalize(question) != pashe_normalize(paraphrase),
        "predicted_question_type": pashe_predict_question_type(question),
        "reference": str(reference),
        "answer_type": str(dataset.dataset[dataset_index].get("answer_type", "Open-ended")),
        "slide_id": str(dataset.dataset[dataset_index].get("slide_id", dataset_index)),
        "greedy": greedy,
        "records": records,
    }

## 3. Dataset-specific clustering backends

WSI-Bench validation compares Exact text with SBERT/BGE cosine thresholds
`0.70`, `0.80`, and `0.90`, plus bidirectional RoBERTa/DeBERTa-NLI
thresholds `0.35`, `0.50`, and `0.65`. Each method's pairwise score matrix is
calculated once per example and reused across thresholds. Official test
constructs only the separately validation-locked overall/open configurations.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

pashe_sbert = SentenceTransformer(PASHE_SBERT_MODEL, device=str(device))
pashe_bge = SentenceTransformer(PASHE_BGE_MODEL, device=str(device))


def pashe_load_nli(model_name):
    tokenizer_nli = AutoTokenizer.from_pretrained(model_name)
    model_nli = AutoModelForSequenceClassification.from_pretrained(
        model_name
    ).to(device).eval()
    entailment_id = next(
        (
            int(index)
            for index, label in model_nli.config.id2label.items()
            if "entail" in str(label).lower()
        ),
        2,
    )
    return tokenizer_nli, model_nli, entailment_id


pashe_roberta_tok, pashe_roberta, pashe_roberta_entail = pashe_load_nli(
    PASHE_ROBERTA_NLI_MODEL
)
pashe_deberta_tok, pashe_deberta, pashe_deberta_entail = pashe_load_nli(
    PASHE_DEBERTA_NLI_MODEL
)


def pashe_unique_answers(answers):
    normalized = [pashe_normalize(answer) for answer in answers]
    unique = list(dict.fromkeys(normalized))
    return normalized, unique


def pashe_exact_clusters(answers):
    normalized, unique = pashe_unique_answers(answers)
    mapping = {answer: index for index, answer in enumerate(unique)}
    return [mapping[answer] for answer in normalized]


def pashe_embedding_cache(answers, encoder):
    normalized, unique = pashe_unique_answers(answers)
    embeddings = encoder.encode(
        unique,
        normalize_embeddings=True,
        batch_size=PASHE_MODEL_BATCH_SIZE,
    )
    matrix = np.asarray(embeddings) @ np.asarray(embeddings).T
    return normalized, unique, matrix


@torch.inference_mode()
def pashe_nli_cache(answers, tokenizer_nli, model_nli, entailment_id):
    normalized, unique = pashe_unique_answers(answers)
    scores = np.eye(len(unique), dtype=float)
    pairs = [
        (i, j)
        for i in range(len(unique))
        for j in range(len(unique))
        if i != j
    ]
    for start in range(0, len(pairs), PASHE_MODEL_BATCH_SIZE):
        batch = pairs[start:start + PASHE_MODEL_BATCH_SIZE]
        encoded = tokenizer_nli(
            [unique[i] for i, _ in batch],
            [unique[j] for _, j in batch],
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        probabilities = torch.softmax(
            model_nli(**encoded).logits,
            dim=-1,
        )[:, entailment_id]
        for (i, j), value in zip(batch, probabilities.cpu().tolist()):
            scores[i, j] = value
    return normalized, unique, scores


def pashe_clusters_from_similarity(cache, threshold, bidirectional=False):
    normalized, unique, matrix = cache
    representatives = []
    unique_cluster_ids = []
    for i in range(len(unique)):
        assigned = None
        for cluster_id, representative in enumerate(representatives):
            forward = matrix[i, representative] >= threshold
            backward = matrix[representative, i] >= threshold
            if forward and (backward if bidirectional else True):
                assigned = cluster_id
                break
        if assigned is None:
            assigned = len(representatives)
            representatives.append(i)
        unique_cluster_ids.append(assigned)
    mapping = dict(zip(unique, unique_cluster_ids))
    return [mapping[item] for item in normalized]
_qa_snne_embedding_encoder = None


def qa_snne_get_embedding_encoder():
    global _qa_snne_embedding_encoder
    if _qa_snne_embedding_encoder is None:
        _qa_snne_embedding_encoder = SentenceTransformer(
            QA_SNNE_EMBEDDING_MODEL, device=str(device)
        )
    return _qa_snne_embedding_encoder


def qa_snne_embedding_alignment(question, answers):
    encoder = qa_snne_get_embedding_encoder()
    embeddings = np.asarray(encoder.encode(
        [str(question)] + [str(answer) for answer in answers],
        normalize_embeddings=True,
        batch_size=PASHE_MODEL_BATCH_SIZE,
    ))
    return embeddings[1:] @ embeddings[0]


# Only the embedding-based QA-SNNE comparator is retained.


## 4. Risk definitions

For condition \(k\), sampled sequence log-probabilities are length- and temperature-calibrated, normalised within that condition, and accumulated by semantic cluster:

\[
p_k(c)=\frac{\sum_{s\in k,\ z(s)=c}\exp(\ell_s/(L_s^\alpha T))}
{\sum_{s\in k}\exp(\ell_s/(L_s^\alpha T))}.
\]

With \(K\) available conditions, PA-SHE uses \(\bar p(c)=K^{-1}\sum_k p_k(c)\) and \(H_{\mathrm{PA-SHE}}=-\sum_c\bar p(c)\log\bar p(c)\). SE uses only \(p_{\mathrm{original}}\). VASE is the Jensen–Shannon divergence between weak and distorted condition distributions. SNNE uses pairwise ROUGE-L among 20 original-input samples; QA-SNNE adds question–answer embedding alignment.

In [ ]:
PASHE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]


def pashe_entropy(probabilities):
    p = np.asarray(probabilities, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log(p + 1e-12)).sum())


def pashe_record_sequence_length(record):
    if "_sequence_length" not in record:
        record["_sequence_length"] = max(1, len(tokenizer.encode(
            str(record.get("answer", "")), add_special_tokens=False
        )))
    return int(record["_sequence_length"])


def pashe_distribution(
    records, cluster_ids, condition, cluster_count,
    length_alpha, weight_temperature,
):
    indices = [i for i, r in enumerate(records) if r["condition"] == condition]
    p = np.zeros(cluster_count, dtype=float)
    if not indices:
        return p
    if length_alpha < 0 or weight_temperature <= 0:
        raise ValueError("Invalid sequence-probability calibration.")
    logps = np.asarray([records[i]["sequence_logprob"] for i in indices])
    lengths = np.asarray([
        pashe_record_sequence_length(records[i]) for i in indices
    ], dtype=float)
    calibrated = logps / np.power(lengths, float(length_alpha))
    calibrated = calibrated / float(weight_temperature)
    weights = np.exp(calibrated - calibrated.max())
    weights /= max(weights.sum(), 1e-12)
    for i, weight in zip(indices, weights):
        p[int(cluster_ids[i])] += float(weight)
    return p / max(p.sum(), 1e-12)


def pashe_signals(
    example, all_cluster_ids, length_alpha, weight_temperature,
):
    records = example["records"]
    record_ids = all_cluster_ids
    cluster_count = max(all_cluster_ids) + 1
    distributions = {
        condition: pashe_distribution(
            records, record_ids, condition, cluster_count,
            length_alpha, weight_temperature,
        )
        for condition in PASHE_CONDITIONS
    }
    available = [p for p in distributions.values() if p.sum() > 0]
    pooled = np.mean(available, axis=0)
    p_original = distributions["original"]
    p_weak, p_distorted = distributions["weak"], distributions["distorted"]
    vase = float(jensenshannon(
        p_weak + 1e-12, p_distorted + 1e-12, base=2.0
    ) ** 2)
    return {
        "vase": vase,
        "semantic_entropy": pashe_entropy(p_original),
        "pa_she": pashe_entropy(pooled),
        "cluster_count": int(cluster_count),
    }


pashe_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def pashe_rouge_l(reference, prediction):
    reference = pashe_normalize(reference)
    prediction = pashe_normalize(prediction)
    return float(pashe_rouge.score(reference, prediction)["rougeL"].fmeasure)

def qa_snne_rouge_similarity_matrix(answers):
    answers = [pashe_normalize(answer) for answer in answers]
    n = len(answers)
    matrix = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i + 1, n):
            forward = pashe_rouge_l(answers[i], answers[j])
            backward = pashe_rouge_l(answers[j], answers[i])
            matrix[i, j] = matrix[j, i] = 0.5 * (forward + backward)
    return matrix


def qa_snne_score(similarity_matrix, alignment_scores=None):
    """Equations (1)-(4) of Carlini et al.; higher means less certain."""
    similarity = np.asarray(similarity_matrix, dtype=np.float64)
    n = similarity.shape[0]
    if similarity.shape != (n, n) or n < 2:
        raise ValueError("SNNE requires a square matrix with at least two answers.")
    if alignment_scores is not None:
        alignment = np.asarray(alignment_scores, dtype=np.float64)
        if alignment.shape != (n,):
            raise ValueError("QA-SNNE alignment scores must match sampled answers.")
        shifted = QA_SNNE_BETA * alignment
        shifted -= shifted.max()
        relevance = np.exp(shifted)
        relevance /= max(relevance.sum(), 1e-12)
        similarity = np.diag(relevance) @ similarity @ np.diag(relevance)

    row_log_sums = []
    for i in range(n):
        values = np.delete(similarity[i], i) / QA_SNNE_TAU
        maximum = float(values.max())
        row_log_sums.append(
            maximum + np.log(np.exp(values - maximum).sum() + 1e-12)
        )
    return float(-np.mean(row_log_sums))


def qa_snne_signals(example, qa_sample_example):
    answers = [str(answer) for answer in qa_sample_example["answers"]]
    if len(answers) != QA_SNNE_NUM_SAMPLES:
        raise ValueError("QA-SNNE sample count does not match configuration.")
    question = str(example["question"])
    similarity = qa_snne_rouge_similarity_matrix(answers)
    embedding_alignment = qa_snne_embedding_alignment(question, answers)
    return {
        "snne": qa_snne_score(similarity),
        "qa_snne_embedding": qa_snne_score(similarity, embedding_alignment),
        "qa_snne_embedding_alignment_mean": float(np.mean(embedding_alignment)),
    }

## 5. Validation sampling and cache

Only validation examples are sampled in this cell. The selected test scope is
not sampled until section 7 has selected and locked a clustering rule. In the
default protocol, that scope is the explicitly reported published-feature subset.

In [ ]:
PASHE_CONDITIONS = ["original", "weak", "distorted", "paraphrase"]
pashe_datasets = {
    "validation": val_dataset,
    "test": test_dataset,
}


def pashe_dataset_signature(dataset, evaluation_size):
    digest = hashlib.sha256()
    for dataset_index in range(evaluation_size):
        raw = dataset.dataset[dataset_index]
        digest.update(str(dataset_index).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["question"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw["answer"]).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw.get("slide_id", "")).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(raw.get("answer_type", "")).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def pashe_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if PASHE_MAX_EXAMPLES is None
        else min(int(PASHE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = PASHE_CHECKPOINT_PATH.stat()
    cache_configuration = {
        "cache_schema_version": PASHE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": pashe_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(PASHE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "samples_per_condition": PASHE_NUM_SAMPLES,
        "maximum_new_tokens": PASHE_MAX_NEW_TOKENS,
        "temperature": PASHE_TEMPERATURE,
        "top_p": PASHE_TOP_P,
        "seed": PASHE_RANDOM_SEED,
        "perturbation_version": PASHE_PERTURBATION_VERSION,
        "conditions": PASHE_CONDITIONS,
    }
    cache_hash = hashlib.sha256(
        json.dumps(cache_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = PASHE_CACHE_DIR / f"{split_name}_samples_{cache_hash}.jsonl"

    cached_examples = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as cache_file:
            for line_number, line in enumerate(cache_file, start=1):
                if not line.strip():
                    continue
                try:
                    example = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete cache line {line_number}: {cache_path}")
                    continue
                dataset_index = int(example.get("dataset_index", -1))
                if example.get("cache_schema_version") != PASHE_CACHE_SCHEMA_VERSION:
                    raise ValueError("PA-SHE sample-cache schema mismatch.")
                if example.get("split") != split_name:
                    raise ValueError("PA-SHE sample-cache split mismatch.")
                if not 0 <= dataset_index < evaluation_size:
                    raise ValueError("Cached dataset index is outside this run.")
                condition_counts = {
                    condition: sum(
                        record.get("condition") == condition
                        for record in example.get("records", [])
                    )
                    for condition in PASHE_CONDITIONS
                }
                if any(
                    count != PASHE_NUM_SAMPLES
                    for count in condition_counts.values()
                ):
                    raise ValueError("Cached condition/sample counts do not match.")
                if dataset_index in cached_examples:
                    raise ValueError("Duplicate dataset index in PA-SHE cache.")
                cached_examples[dataset_index] = example

    pending_indices = [
        index for index in range(evaluation_size)
        if index not in cached_examples
    ]
    print({
        "split": split_name,
        "sample_cache": str(cache_path),
        "cached_examples": len(cached_examples),
        "pending_examples": len(pending_indices),
    })

    split_seed_offset = 0 if split_name == "validation" else 10_000_000
    with cache_path.open("a", encoding="utf-8") as cache_file:
        for dataset_index in tqdm(
            pending_indices,
            desc=f"Frozen-VQA sampling: {split_name}",
        ):
            example_seed = PASHE_RANDOM_SEED + split_seed_offset + dataset_index * 1009
            random.seed(example_seed)
            np.random.seed(example_seed)
            torch.manual_seed(example_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(example_seed)
            example = pashe_collect_example(
                dataset=dataset,
                dataset_index=dataset_index,
                split_name=split_name,
            )
            cache_file.write(json.dumps(example, ensure_ascii=False) + "\n")
            cache_file.flush()
            cached_examples[dataset_index] = example

    return (
        [cached_examples[index] for index in range(evaluation_size)],
        cache_path,
    )


# Selection starts with validation only. Test sampling occurs in section 7,
# after PASHE_LOCKED_CLUSTERING has been assigned.
def qa_snne_collect_split(split_name, dataset):
    evaluation_size = (
        len(dataset)
        if PASHE_MAX_EXAMPLES is None
        else min(int(PASHE_MAX_EXAMPLES), len(dataset))
    )
    checkpoint_stat = PASHE_CHECKPOINT_PATH.stat()
    configuration = {
        "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
        "split": split_name,
        "evaluation_size": evaluation_size,
        "dataset_signature": pashe_dataset_signature(dataset, evaluation_size),
        "checkpoint_path": str(PASHE_CHECKPOINT_PATH),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "num_samples": QA_SNNE_NUM_SAMPLES,
        "maximum_new_tokens": PASHE_MAX_NEW_TOKENS,
        "temperature": QA_SNNE_TEMPERATURE,
        "top_k": QA_SNNE_TOP_K,
        "top_p": QA_SNNE_TOP_P,
        "seed": PASHE_RANDOM_SEED,
        "input_condition": "original image and original question",
    }
    cache_hash = hashlib.sha256(
        json.dumps(configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    cache_path = PASHE_CACHE_DIR / f"{split_name}_qa_snne_samples_{cache_hash}.jsonl"
    cached = {}
    if cache_path.exists():
        with cache_path.open("r", encoding="utf-8") as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f"Ignoring incomplete QA-SNNE cache line {line_number}")
                    continue
                index = int(record.get("dataset_index", -1))
                if record.get("cache_schema_version") != QA_SNNE_CACHE_SCHEMA_VERSION:
                    raise ValueError("QA-SNNE sample-cache schema mismatch.")
                if record.get("split") != split_name or not 0 <= index < evaluation_size:
                    raise ValueError("QA-SNNE sample-cache split/index mismatch.")
                if len(record.get("answers", [])) != QA_SNNE_NUM_SAMPLES:
                    raise ValueError("QA-SNNE cached sample count mismatch.")
                if index in cached:
                    raise ValueError("Duplicate index in QA-SNNE sample cache.")
                cached[index] = record

    pending = [index for index in range(evaluation_size) if index not in cached]
    print({
        "split": split_name,
        "qa_snne_sample_cache": str(cache_path),
        "cached_examples": len(cached),
        "pending_examples": len(pending),
        "samples_per_example": QA_SNNE_NUM_SAMPLES,
    })
    split_offset = 30_000_000 if split_name == "validation" else 40_000_000
    with cache_path.open("a", encoding="utf-8") as handle:
        for index in tqdm(pending, desc=f"QA-SNNE sampling: {split_name}"):
            seed = PASHE_RANDOM_SEED + split_offset + index * 1013
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
            image, question, _ = dataset[index]
            record = {
                "cache_schema_version": QA_SNNE_CACHE_SCHEMA_VERSION,
                "split": split_name,
                "dataset_index": int(index),
                "question": str(question),
                "answers": qa_snne_generate_samples(
                    image, question, QA_SNNE_NUM_SAMPLES
                ),
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            cached[index] = record
    return [cached[index] for index in range(evaluation_size)], cache_path

pashe_validation_examples, pashe_validation_sample_cache_path = pashe_collect_split(
    "validation",
    pashe_datasets["validation"],
)
pashe_examples_by_split = {"validation": pashe_validation_examples}
pashe_sample_cache_paths = {"validation": pashe_validation_sample_cache_path}
qa_snne_validation_examples, qa_snne_validation_sample_cache_path = (
    qa_snne_collect_split("validation", pashe_datasets["validation"])
)
qa_snne_examples_by_split = {"validation": qa_snne_validation_examples}
qa_snne_sample_cache_paths = {
    "validation": qa_snne_validation_sample_cache_path
}

print({
    "validation_examples": len(pashe_validation_examples),
    "test_sampled_before_selection": False,
})
display(pd.DataFrame([{
    "split": example["split"],
    "index": example["dataset_index"],
    "question": example["question"],
    "reference": example["reference"],
    "greedy": example["greedy"],
} for example in pashe_validation_examples[:5]]))

## 6. Validation candidate features

Build validation features for the predeclared 13-candidate grid: Exact text
and three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
The same cache also stores VASE, SNNE, and embedding-based QA-SNNE.
No test feature exists yet.

In [ ]:
PASHE_VALIDATION_FEATURE_CONFIGURATIONS = list(dict.fromkeys([
    PASHE_CONTROL_CLUSTERING,
    *PASHE_SELECTION_CANDIDATES,
]))


def pashe_clusters_for_configurations(answers, clustering_configurations):
    clustering_configurations = list(clustering_configurations)
    unsupported = set(clustering_configurations) - set(PASHE_SELECTION_CANDIDATES)
    if unsupported:
        raise ValueError(
            "Unsupported WSI-Bench clustering configuration(s): "
            + ", ".join(sorted(unsupported))
        )

    requested = set(clustering_configurations)
    configurations = []
    if "Exact text" in requested:
        configurations.append(("Exact text", pashe_exact_clusters(answers)))

    embedding_specs = [
        ("SBERT", PASHE_SBERT_THRESHOLDS, pashe_sbert),
        ("BGE", PASHE_BGE_THRESHOLDS, pashe_bge),
    ]
    for method_name, thresholds, encoder in embedding_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = pashe_embedding_cache(answers, encoder)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    pashe_clusters_from_similarity(
                        score_cache, threshold, bidirectional=False
                    ),
                ))

    nli_specs = [
        (
            "RoBERTa-NLI", PASHE_ROBERTA_NLI_THRESHOLDS,
            pashe_roberta_tok, pashe_roberta, pashe_roberta_entail,
        ),
        (
            "DeBERTa-NLI", PASHE_DEBERTA_NLI_THRESHOLDS,
            pashe_deberta_tok, pashe_deberta, pashe_deberta_entail,
        ),
    ]
    for method_name, thresholds, tok, mdl, entail_id in nli_specs:
        requested_thresholds = [
            threshold for threshold in thresholds
            if f"{method_name}@{threshold:.2f}" in requested
        ]
        if requested_thresholds:
            score_cache = pashe_nli_cache(answers, tok, mdl, entail_id)
            for threshold in requested_thresholds:
                configuration_name = f"{method_name}@{threshold:.2f}"
                configurations.append((
                    configuration_name,
                    pashe_clusters_from_similarity(
                        score_cache, threshold, bidirectional=True
                    ),
                ))
    if {name for name, _ in configurations} != set(clustering_configurations):
        raise ValueError("Failed to construct every requested clustering.")
    return configurations


def pashe_build_feature_frame(
    split_name,
    examples,
    sample_cache_path,
    clustering_configurations,
    weighting_configurations,
):
    clustering_configurations = list(clustering_configurations)
    weighting_configurations = [tuple(item) for item in weighting_configurations]
    feature_configuration = {
        "feature_schema_version": PASHE_FEATURE_SCHEMA_VERSION,
        "split": split_name,
        "sample_cache": sample_cache_path.name,
        "clustering_configurations": clustering_configurations,
        "weighting_configurations": weighting_configurations,
        "metric_normalization_version": 1,
        "question_router_version": PASHE_QUESTION_ROUTER_VERSION,
        "sbert_model": PASHE_SBERT_MODEL,
        "sbert_thresholds": PASHE_SBERT_THRESHOLDS,
        "bge_model": PASHE_BGE_MODEL,
        "bge_thresholds": PASHE_BGE_THRESHOLDS,
        "roberta_nli_model": PASHE_ROBERTA_NLI_MODEL,
        "roberta_nli_thresholds": PASHE_ROBERTA_NLI_THRESHOLDS,
        "deberta_nli_model": PASHE_DEBERTA_NLI_MODEL,
        "deberta_nli_thresholds": PASHE_DEBERTA_NLI_THRESHOLDS,
        "qa_snne_sample_cache": qa_snne_sample_cache_paths[split_name].name,
        "qa_snne_num_samples": QA_SNNE_NUM_SAMPLES,
        "qa_snne_beta": QA_SNNE_BETA,
        "qa_snne_tau": QA_SNNE_TAU,
        "qa_snne_embedding_model": QA_SNNE_EMBEDDING_MODEL,
    }
    feature_hash = hashlib.sha256(
        json.dumps(feature_configuration, sort_keys=True).encode("utf-8")
    ).hexdigest()[:12]
    feature_path = PASHE_CACHE_DIR / f"{split_name}_features_{feature_hash}.csv"

    if feature_path.exists():
        frame = pd.read_csv(feature_path, keep_default_na=False)
        expected_rows = (
            len(examples) * len(clustering_configurations)
            * len(weighting_configurations)
        )
        required_columns = {
            "split", "dataset_index", "clustering", "rougeL", "reference",
            "prediction", "answer_type", "predicted_question_type",
            "length_alpha", "weight_temperature", "vase",
            "semantic_entropy", "pa_she", "cluster_count",
            "snne", "qa_snne_embedding",
        }
        if required_columns - set(frame.columns):
            raise ValueError("Cached WSI-Bench features are incomplete.")
        if len(frame) != expected_rows:
            raise ValueError("Cached WSI-Bench feature row count is incorrect.")
        if set(frame["clustering"]) != set(clustering_configurations):
            raise ValueError("Cached WSI-Bench clustering set is incorrect.")
        print(f"Loaded {split_name} semantic features from {feature_path}")
        return frame, feature_path

    feature_rows = []
    for example in tqdm(examples, desc=f"Semantic clustering: {split_name}"):
        answers = [
            record["answer"] for record in example["records"]
        ]
        qa_sample_example = qa_snne_examples_by_split[split_name][
            int(example["dataset_index"])
        ]
        qa_uncertainty = qa_snne_signals(example, qa_sample_example)
        configurations = pashe_clusters_for_configurations(
            answers,
            clustering_configurations,
        )
        rouge_l = pashe_rouge_l(example["reference"], example["greedy"])
        answer_type = str(example.get("answer_type", "Open-ended"))
        for clustering, cluster_ids in configurations:
            for length_alpha, weight_temperature in weighting_configurations:
                row = {
                "split": split_name,
                "dataset_index": int(example["dataset_index"]),
                    "clustering": clustering,
                    "length_alpha": float(length_alpha),
                    "weight_temperature": float(weight_temperature),
                    "rougeL": rouge_l,
                    "reference": example["reference"],
                    "prediction": example["greedy"],
                    "answer_type": answer_type,
                    "predicted_question_type": pashe_predict_question_type(
                        example["question"]
                    ),
                }
                row.update(pashe_signals(
                    example, cluster_ids, length_alpha, weight_temperature
                ))
                row.update(qa_uncertainty)
                feature_rows.append(row)

    frame = pd.DataFrame(feature_rows)
    frame.to_csv(feature_path, index=False)
    print(f"Saved {split_name} semantic features to {feature_path}")
    return frame, feature_path


pashe_validation_features, pashe_validation_feature_cache_path = (
    pashe_build_feature_frame(
        "validation",
        pashe_validation_examples,
        pashe_validation_sample_cache_path,
        PASHE_VALIDATION_FEATURE_CONFIGURATIONS,
        PASHE_WEIGHTING_CANDIDATES,
    )
)
display(pashe_validation_features.head())
print({
    "dataset": "WSI-Bench",
    "selection_split": "validation",
    "selection_candidates": PASHE_SELECTION_CANDIDATES,
    "test_features_built_before_selection": False,
})

## 7. Question-type-routed validation selection and locked test safety

WSI-Bench contains both descriptive open questions and closed questions. One
clustering backend is selected on all validation examples for overall reporting,
and another is selected on open-ended validation examples. Both choices are locked before test sampling.
The published-feature test subset is evaluated once with those locked choices.

In [ ]:
PASHE_RISK_COLUMNS = {
    "VASE": "vase",
    "SE": "semantic_entropy",
    "SNNE": "snne",
    "QA-SNNE · Embedding": "qa_snne_embedding",
    "PA-SHE": "pa_she",
}


def pashe_safe_metrics(labels, scores):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    if labels.shape != scores.shape:
        raise ValueError("Labels and uncertainty scores must align.")
    if not np.isfinite(scores).all():
        raise ValueError("Uncertainty scores contain NaN or infinity.")
    if np.unique(labels).size < 2:
        return np.nan, np.nan
    return roc_auc_score(labels, scores), average_precision_score(labels, scores)


# Select route-specific clustering and probability calibration on validation only.
def pashe_select_configuration(validation_features, selection_subset):
    expected_examples = validation_features["dataset_index"].nunique()
    rows = []
    for clustering_order, clustering in enumerate(PASHE_SELECTION_CANDIDATES):
        for weighting_order, (length_alpha, weight_temperature) in enumerate(
            PASHE_WEIGHTING_CANDIDATES
        ):
            candidate_order = (
                clustering_order * len(PASHE_WEIGHTING_CANDIDATES) + weighting_order
            )
            group = validation_features[
                (validation_features["clustering"] == clustering)
                & np.isclose(validation_features["length_alpha"], length_alpha)
                & np.isclose(
                    validation_features["weight_temperature"],
                    weight_temperature,
                )
            ].sort_values("dataset_index")
            if len(group) != expected_examples:
                raise ValueError(
                    f"{selection_subset} validation is incomplete for "
                    f"{clustering}, alpha={length_alpha}, T={weight_temperature}."
                )
            failures = (
                group["rougeL"].to_numpy() < PASHE_PRIMARY_LABEL_THRESHOLD
            ).astype(int)
            auroc, auprc = pashe_safe_metrics(failures, group["pa_she"])
            rows.append({
                "selection_split": "slide-disjoint validation",
                "selection_subset": selection_subset,
                "test_used_for_selection": False,
                "candidate_order": candidate_order,
                "label_threshold": PASHE_PRIMARY_LABEL_THRESHOLD,
                "clustering": clustering,
                "length_alpha": float(length_alpha),
                "weight_temperature": float(weight_temperature),
                "examples": len(group),
                "failure_prevalence": failures.mean(),
                "validation_AUROC": auroc,
                "validation_AUPRC": auprc,
            })
    selection = pd.DataFrame(rows)
    ranked = selection.sort_values(
        ["validation_AUROC", "validation_AUPRC", "candidate_order"],
        ascending=[False, False, True],
        kind="mergesort",
    )
    if ranked.empty or pd.isna(ranked.iloc[0]["validation_AUROC"]):
        raise ValueError(
            f"Validation labels cannot select a configuration for {selection_subset}."
        )
    best = ranked.iloc[0]
    return (
        selection,
        str(best["clustering"]),
        float(best["length_alpha"]),
        float(best["weight_temperature"]),
    )

closed_validation_features = pashe_validation_features[
    pashe_validation_features["predicted_question_type"] == "Closed"
].copy()
if closed_validation_features.empty:
    raise ValueError("Question router predicted no closed validation questions.")
(
    pashe_validation_selection,
    PASHE_LOCKED_CLUSTERING,
    PASHE_LOCKED_LENGTH_ALPHA,
    PASHE_LOCKED_WEIGHT_TEMPERATURE,
) = pashe_select_configuration(closed_validation_features, "Predicted closed")

open_validation_features = pashe_validation_features[
    pashe_validation_features["predicted_question_type"] == "Open"
].copy()
if open_validation_features.empty:
    raise ValueError("WSI-Bench validation contains no open-ended examples.")
(
    pashe_open_validation_selection,
    PASHE_OPEN_LOCKED_CLUSTERING,
    PASHE_OPEN_LOCKED_LENGTH_ALPHA,
    PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
) = pashe_select_configuration(open_validation_features, "Predicted open")

# Select the QA-SNNE alignment variant independently on validation only.
qa_validation_base = pashe_validation_features[
    (pashe_validation_features["clustering"] == PASHE_CONTROL_CLUSTERING)
    & np.isclose(pashe_validation_features["length_alpha"], LENGTH_ALPHA_GRID[0])
    & np.isclose(
        pashe_validation_features["weight_temperature"],
        WEIGHT_TEMPERATURE_GRID[0],
    )
].sort_values("dataset_index")
qa_failures = (
    qa_validation_base["rougeL"].to_numpy() < PASHE_PRIMARY_LABEL_THRESHOLD
).astype(int)
qa_selection_rows = []
for variant_order, (variant, column) in enumerate(QA_SNNE_VARIANTS.items()):
    auroc, auprc = pashe_safe_metrics(qa_failures, qa_validation_base[column])
    qa_selection_rows.append({
        "selection_split": "validation",
        "selection_subset": "All",
        "test_used_for_selection": False,
        "variant_order": variant_order,
        "label_threshold": PASHE_PRIMARY_LABEL_THRESHOLD,
        "variant": variant,
        "column": column,
        "examples": len(qa_validation_base),
        "failure_prevalence": qa_failures.mean(),
        "validation_AUROC": auroc,
        "validation_AUPRC": auprc,
    })
qa_snne_validation_selection = pd.DataFrame(qa_selection_rows)
qa_ranked = qa_snne_validation_selection.sort_values(
    ["validation_AUROC", "validation_AUPRC", "variant_order"],
    ascending=[False, False, True],
    kind="mergesort",
)
if qa_ranked.empty or pd.isna(qa_ranked.iloc[0]["validation_AUROC"]):
    raise ValueError("Validation labels cannot select a QA-SNNE variant.")
QA_SNNE_LOCKED_VARIANT = str(qa_ranked.iloc[0]["variant"])
QA_SNNE_LOCKED_COLUMN = str(qa_ranked.iloc[0]["column"])
QA_SNNE_LOCKED_METHOD = f"QA-SNNE · {QA_SNNE_LOCKED_VARIANT}"

selection_path = PASHE_CACHE_DIR / "validation_selected_clustering.json"
selection_payload = {
    "dataset": "WSI-Bench",
    "selection_split": "slide-disjoint validation",
    "test_used_for_selection": False,
    "primary_label_definition": "ROUGE-L < 0.50",
    "selection_metric": "AUROC; AUPRC tie-breaker; candidate order final tie-breaker",
    "eligible_candidates": PASHE_SELECTION_CANDIDATES,
    "length_alpha_grid": LENGTH_ALPHA_GRID,
    "weight_temperature_grid": WEIGHT_TEMPERATURE_GRID,
    "question_router_version": PASHE_QUESTION_ROUTER_VERSION,
    "routing_input": "question text only; reference answer excluded",
    "overall_locked_clustering": PASHE_LOCKED_CLUSTERING,
    "closed_route_locked_clustering": PASHE_LOCKED_CLUSTERING,
    "closed_route_locked_length_alpha": PASHE_LOCKED_LENGTH_ALPHA,
    "closed_route_locked_weight_temperature": PASHE_LOCKED_WEIGHT_TEMPERATURE,
    "open_ended_locked_clustering": PASHE_OPEN_LOCKED_CLUSTERING,
    "open_route_locked_length_alpha": PASHE_OPEN_LOCKED_LENGTH_ALPHA,
    "open_route_locked_weight_temperature": PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
    "qa_snne_locked_variant": QA_SNNE_LOCKED_VARIANT,
    "qa_snne_locked_column": QA_SNNE_LOCKED_COLUMN,
    "qa_snne_selection_metric": (
        "AUROC; AUPRC tie-breaker; variant order final tie-breaker"
    ),
    "qa_snne_candidates": qa_snne_validation_selection.drop(
        columns="variant_order"
    ).to_dict(orient="records"),
    "overall_candidates": pashe_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
    "open_ended_candidates": pashe_open_validation_selection.drop(
        columns="candidate_order"
    ).to_dict(orient="records"),
}
selection_path.write_text(
    json.dumps(selection_payload, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
pashe_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_clustering_selection.csv", index=False
)
pashe_open_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_open_ended_clustering_selection.csv",
    index=False,
)
qa_snne_validation_selection.to_csv(
    PASHE_CACHE_DIR / "validation_qa_snne_selection.csv",
    index=False,
)

# Only now sample the selected test scope and build routed/ablation features.
PASHE_RAW_LENGTH_ALPHA = 0.0
PASHE_RAW_WEIGHT_TEMPERATURE = 1.0
PASHE_RAW_PA_SHE_METHOD = "PA-SHE · Raw sequence log-probability"
qa_snne_test_examples, qa_snne_test_sample_cache_path = (
    qa_snne_collect_split("test", pashe_datasets["test"])
)
qa_snne_examples_by_split["test"] = qa_snne_test_examples
qa_snne_sample_cache_paths["test"] = qa_snne_test_sample_cache_path

pashe_test_examples, pashe_test_sample_cache_path = pashe_collect_split(
    "test", pashe_datasets["test"]
)
pashe_examples_by_split["test"] = pashe_test_examples
pashe_sample_cache_paths["test"] = pashe_test_sample_cache_path
locked_configuration_union = list(dict.fromkeys([
    PASHE_LOCKED_CLUSTERING,
    PASHE_OPEN_LOCKED_CLUSTERING,
]))
pashe_test_features, pashe_test_feature_cache_path = pashe_build_feature_frame(
    "test",
    pashe_test_examples,
    pashe_test_sample_cache_path,
    locked_configuration_union,
    list(dict.fromkeys([
        (PASHE_LOCKED_LENGTH_ALPHA, PASHE_LOCKED_WEIGHT_TEMPERATURE),
        (PASHE_OPEN_LOCKED_LENGTH_ALPHA, PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE),
        (PASHE_RAW_LENGTH_ALPHA, PASHE_RAW_WEIGHT_TEMPERATURE),
    ])),
)
pashe_locked_test_features = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_LOCKED_CLUSTERING)
    & np.isclose(pashe_test_features["length_alpha"], PASHE_LOCKED_LENGTH_ALPHA)
    & np.isclose(
        pashe_test_features["weight_temperature"],
        PASHE_LOCKED_WEIGHT_TEMPERATURE,
    )
    & (pashe_test_features["predicted_question_type"] == "Closed")
].sort_values("dataset_index").copy()
if pashe_locked_test_features["dataset_index"].duplicated().any():
    raise ValueError("Locked WSI-Bench test features contain duplicate questions.")
pashe_open_locked_test_features = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING)
    & np.isclose(
        pashe_test_features["length_alpha"], PASHE_OPEN_LOCKED_LENGTH_ALPHA
    )
    & np.isclose(
        pashe_test_features["weight_temperature"],
        PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
    )
    & (pashe_test_features["predicted_question_type"] == "Open")
].sort_values("dataset_index").copy()
if pashe_open_locked_test_features.empty:
    raise ValueError("WSI-Bench test contains no open-ended examples.")
if pashe_open_locked_test_features["dataset_index"].duplicated().any():
    raise ValueError("Open-ended locked WSI-Bench features contain duplicates.")
expected_test_examples = pashe_test_features["dataset_index"].nunique()
pashe_routed_test_features = pd.concat(
    [pashe_locked_test_features, pashe_open_locked_test_features],
    ignore_index=True,
).sort_values("dataset_index")
if (
    len(pashe_routed_test_features) != expected_test_examples
    or pashe_routed_test_features["dataset_index"].duplicated().any()
):
    raise ValueError("Question-type routing must cover every test example once.")

closed_validation_locked = closed_validation_features[
    (closed_validation_features["clustering"] == PASHE_LOCKED_CLUSTERING)
    & np.isclose(closed_validation_features["length_alpha"], PASHE_LOCKED_LENGTH_ALPHA)
    & np.isclose(
        closed_validation_features["weight_temperature"],
        PASHE_LOCKED_WEIGHT_TEMPERATURE,
    )
].sort_values("dataset_index")
open_validation_locked = open_validation_features[
    (open_validation_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING)
    & np.isclose(open_validation_features["length_alpha"], PASHE_OPEN_LOCKED_LENGTH_ALPHA)
    & np.isclose(
        open_validation_features["weight_temperature"],
        PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
    )
].sort_values("dataset_index")


def pashe_validation_percentile(validation_scores, query_scores):
    reference = np.sort(np.asarray(validation_scores, dtype=float))
    query = np.asarray(query_scores, dtype=float)
    if reference.size == 0:
        raise ValueError("Cannot calibrate from an empty validation route.")
    return np.searchsorted(reference, query, side="right") / reference.size


for _, risk_column in PASHE_RISK_COLUMNS.items():
    calibrated_column = f"calibrated_{risk_column}"
    pashe_locked_test_features[calibrated_column] = pashe_validation_percentile(
        closed_validation_locked[risk_column], pashe_locked_test_features[risk_column]
    )
    pashe_open_locked_test_features[calibrated_column] = pashe_validation_percentile(
        open_validation_locked[risk_column],
        pashe_open_locked_test_features[risk_column],
    )
pashe_routed_test_features = pd.concat(
    [pashe_locked_test_features, pashe_open_locked_test_features],
    ignore_index=True,
).sort_values("dataset_index")
pashe_features = pashe_routed_test_features

raw_closed_validation = closed_validation_features[
    (closed_validation_features["clustering"] == PASHE_LOCKED_CLUSTERING)
    & np.isclose(closed_validation_features["length_alpha"], PASHE_RAW_LENGTH_ALPHA)
    & np.isclose(
        closed_validation_features["weight_temperature"],
        PASHE_RAW_WEIGHT_TEMPERATURE,
    )
].sort_values("dataset_index")
raw_open_validation = open_validation_features[
    (open_validation_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING)
    & np.isclose(open_validation_features["length_alpha"], PASHE_RAW_LENGTH_ALPHA)
    & np.isclose(
        open_validation_features["weight_temperature"],
        PASHE_RAW_WEIGHT_TEMPERATURE,
    )
].sort_values("dataset_index")
raw_closed_test = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_LOCKED_CLUSTERING)
    & np.isclose(pashe_test_features["length_alpha"], PASHE_RAW_LENGTH_ALPHA)
    & np.isclose(pashe_test_features["weight_temperature"], PASHE_RAW_WEIGHT_TEMPERATURE)
    & (pashe_test_features["predicted_question_type"] == "Closed")
].sort_values("dataset_index").copy()
raw_open_test = pashe_test_features[
    (pashe_test_features["clustering"] == PASHE_OPEN_LOCKED_CLUSTERING)
    & np.isclose(pashe_test_features["length_alpha"], PASHE_RAW_LENGTH_ALPHA)
    & np.isclose(pashe_test_features["weight_temperature"], PASHE_RAW_WEIGHT_TEMPERATURE)
    & (pashe_test_features["predicted_question_type"] == "Open")
].sort_values("dataset_index").copy()
if raw_closed_validation.empty or raw_open_validation.empty:
    raise ValueError("Raw-weight validation ablation is incomplete.")
raw_closed_test["calibrated_pa_she"] = pashe_validation_percentile(
    raw_closed_validation["pa_she"], raw_closed_test["pa_she"]
)
raw_open_test["calibrated_pa_she"] = pashe_validation_percentile(
    raw_open_validation["pa_she"], raw_open_test["pa_she"]
)
pashe_raw_routed_test_features = pd.concat(
    [raw_closed_test, raw_open_test], ignore_index=True
).sort_values("dataset_index")
if (
    len(pashe_raw_routed_test_features) != expected_test_examples
    or pashe_raw_routed_test_features["dataset_index"].duplicated().any()
):
    raise ValueError("Raw-weight routing must cover every test example once.")

evaluation_subsets = {
    "All": pashe_routed_test_features,
    "Closed-ended": pashe_routed_test_features[
        pashe_routed_test_features["answer_type"] == "Closed-ended"
    ],
    "Open-ended": pashe_routed_test_features[
        pashe_routed_test_features["answer_type"] == "Open-ended"
    ],
}

pashe_result_rows = []
for evaluation_subset, subset_features in evaluation_subsets.items():
    for label_threshold in PASHE_LABEL_THRESHOLDS:
        failures = (
            subset_features["rougeL"].to_numpy() < label_threshold
        ).astype(int)
        for method, column in PASHE_RISK_COLUMNS.items():
            score_column = f"calibrated_{column}"
            auroc, auprc = pashe_safe_metrics(failures, subset_features[score_column])
            pashe_result_rows.append({
                "evaluation_split": WSIBENCH_TEST_SCOPE_LABEL,
                "evaluation_subset": evaluation_subset,
                "examples": len(subset_features),
                "label_threshold": label_threshold,
                "failure_prevalence": failures.mean() if len(failures) else np.nan,
                "clustering": "Question-type routed",
                "length_alpha": np.nan,
                "weight_temperature": np.nan,
                "method": method,
                "AUROC": auroc,
                "AUPRC": auprc,
            })

raw_evaluation_subsets = {
    "All": pashe_raw_routed_test_features,
    "Closed-ended": pashe_raw_routed_test_features[
        pashe_raw_routed_test_features["answer_type"] == "Closed-ended"
    ],
    "Open-ended": pashe_raw_routed_test_features[
        pashe_raw_routed_test_features["answer_type"] == "Open-ended"
    ],
}
for evaluation_subset, subset_features in raw_evaluation_subsets.items():
    for label_threshold in PASHE_LABEL_THRESHOLDS:
        failures = (
            subset_features["rougeL"].to_numpy() < label_threshold
        ).astype(int)
        auroc, auprc = pashe_safe_metrics(
            failures, subset_features["calibrated_pa_she"]
        )
        pashe_result_rows.append({
            "evaluation_split": WSIBENCH_TEST_SCOPE_LABEL,
            "evaluation_subset": evaluation_subset,
            "examples": len(subset_features),
            "label_threshold": label_threshold,
            "failure_prevalence": failures.mean() if len(failures) else np.nan,
            "clustering": "Question-type routed",
            "length_alpha": PASHE_RAW_LENGTH_ALPHA,
            "weight_temperature": PASHE_RAW_WEIGHT_TEMPERATURE,
            "method": PASHE_RAW_PA_SHE_METHOD,
            "AUROC": auroc,
            "AUPRC": auprc,
        })

pashe_results = pd.DataFrame(pashe_result_rows)
pashe_primary_results = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)
pashe_label_sensitivity = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & (pashe_results["method"] == "PA-SHE")
].sort_values("label_threshold")
pashe_open_ended_results = pashe_results[
    pashe_results["evaluation_subset"] == "Open-ended"
].copy()
pashe_open_ended_primary_results = pashe_open_ended_results[
    np.isclose(pashe_open_ended_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].sort_values(["AUROC", "AUPRC"], ascending=False)
pashe_open_ended_label_sensitivity = pashe_open_ended_results[
    pashe_open_ended_results["method"] == "PA-SHE"
].sort_values("label_threshold")
pashe_closed_ended_results = pashe_results[
    pashe_results["evaluation_subset"] == "Closed-ended"
].copy()

pashe_results.to_csv(PASHE_CACHE_DIR / "locked_test_safety_results.csv", index=False)
pashe_open_ended_results.to_csv(
    PASHE_CACHE_DIR / "locked_test_open_ended_safety_results.csv", index=False
)
pashe_closed_ended_results.to_csv(
    PASHE_CACHE_DIR / "locked_test_closed_ended_safety_results.csv", index=False
)

print("VALIDATION-ONLY QA-SNNE variant selection:")
display(qa_snne_validation_selection.drop(columns="variant_order").round(4))
print("Locked QA-SNNE variant before test sampling:", QA_SNNE_LOCKED_VARIANT)
print("VALIDATION-ONLY WSI-Bench clustering selection:")
display(pashe_validation_selection.drop(columns="candidate_order").round(4))
print("VALIDATION-ONLY open-ended clustering selection:")
display(
    pashe_open_validation_selection.drop(columns="candidate_order").round(4)
)
print("Closed-route locked configuration:", {
    "clustering": PASHE_LOCKED_CLUSTERING,
    "length_alpha": PASHE_LOCKED_LENGTH_ALPHA,
    "weight_temperature": PASHE_LOCKED_WEIGHT_TEMPERATURE,
})
print("Open-route locked configuration:", {
    "clustering": PASHE_OPEN_LOCKED_CLUSTERING,
    "length_alpha": PASHE_OPEN_LOCKED_LENGTH_ALPHA,
    "weight_temperature": PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE,
})
print(f"{WSIBENCH_TEST_SCOPE_LABEL.upper()} primary results: failure = ROUGE-L < 0.50")
display(pashe_primary_results.round(4))
print("OFFICIAL TEST open-ended primary results")
display(pashe_open_ended_primary_results.round(4))
print("PA-SHE test label sensitivity; clustering remains locked")
display(pashe_label_sensitivity.round(4))
print("Open-ended PA-SHE label sensitivity; clustering remains locked")
display(pashe_open_ended_label_sensitivity.round(4))
print("Saved validation selection protocol:", selection_path)

## 8. Validation diagnostic and locked test comparisons

The first plot shows validation-only clustering and probability-calibration
selection for predicted closed and predicted open routes. Subsequent plots use
those locked routes on the held-out published-feature test subset.

In [ ]:
# Validation-only predicted-closed and predicted-open selections.
validation_plots = [
    ("Predicted-closed validation", pashe_validation_selection),
    ("Predicted-open validation", pashe_open_validation_selection),
]
fig, axes = plt.subplots(2, 2, figsize=(22, 11))
for row_index, (subset_label, selection_frame) in enumerate(validation_plots):
    validation_plot = selection_frame.sort_values("candidate_order")
    axes[row_index, 0].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUROC"],
        color="#4c78a8",
    )
    axes[row_index, 1].bar(
        validation_plot["clustering"],
        validation_plot["validation_AUPRC"],
        color="#f58518",
    )
    axes[row_index, 0].set_title(f"{subset_label}: PA-SHE AUROC")
    axes[row_index, 1].set_title(f"{subset_label}: PA-SHE AUPRC")
    for axis in axes[row_index]:
        axis.set_ylim(0, 1)
        axis.tick_params(axis="x", rotation=45)
        axis.grid(axis="y", alpha=0.25)
fig.suptitle("WSI-Bench routed validation locks: ROUGE-L < 0.50")
plt.tight_layout()
plt.show()

# Official-test label sensitivity; overall clustering stays locked.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(
    pashe_label_sensitivity["label_threshold"],
    pashe_label_sensitivity["AUROC"], marker="o"
)
axes[1].plot(
    pashe_label_sensitivity["label_threshold"],
    pashe_label_sensitivity["AUPRC"], marker="o", color="#f58518"
)
axes[0].set_title("Official test AUROC sensitivity")
axes[1].set_title("Official test AUPRC sensitivity")
for axis in axes:
    axis.set_xlabel("ROUGE-L failure-label threshold")
    axis.set_ylim(0, 1)
    axis.set_xticks(PASHE_LABEL_THRESHOLDS)
    axis.grid(alpha=0.25)
fig.suptitle("PA-SHE with validation-locked question-type routing")
plt.tight_layout()
plt.show()

primary_comparison = pashe_primary_results.sort_values("AUROC")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(primary_comparison["method"], primary_comparison["AUROC"])
axes[1].barh(
    primary_comparison["method"], primary_comparison["AUPRC"], color="#d17a22"
)
axes[0].set_title("Official test AUROC")
axes[1].set_title("Official test AUPRC")
for axis in axes:
    axis.set_xlim(0, 1)
    axis.grid(axis="x", alpha=0.25)
fig.suptitle(
    f"Validation-locked WSI-Bench clustering; "
    f"failure = ROUGE-L < {PASHE_PRIMARY_LABEL_THRESHOLD:.2f}"
)
plt.tight_layout()
plt.show()

## 9. Locked selective prediction and audit

These analyses use only official-test features from the validation-locked
clustering configuration.

In [ ]:
primary_features = pashe_routed_test_features.copy()
rejection_fractions = np.linspace(0, 0.50, 11)
curve_rows = []

for method, column in PASHE_RISK_COLUMNS.items():
    ordered = primary_features.sort_values(
        f"calibrated_{column}", ascending=True
    )
    for fraction in rejection_fractions:
        retained_count = max(1, int(round(len(ordered) * (1.0 - fraction))))
        retained = ordered.iloc[:retained_count]
        curve_rows.append({
            "method": method,
            "rejected_fraction": fraction,
            "retained_ROUGE-L": retained["rougeL"].mean(),
        })

pashe_rejection = pd.DataFrame(curve_rows)
plt.figure(figsize=(11, 6))
for method, group in pashe_rejection.groupby("method", sort=False):
    plt.plot(
        group["rejected_fraction"],
        group["retained_ROUGE-L"],
        marker="o",
        label=method,
    )
plt.xlabel("Fraction rejected as high risk")
plt.ylabel("Mean ROUGE-L among retained answers")
plt.title("WSI-Bench selective prediction: validation-locked routing")
plt.grid(alpha=0.25)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

audit_columns = [
    "dataset_index", "answer_type", "reference", "prediction", "rougeL",
    "pa_she", "semantic_entropy",
]
print("Highest PA-SHE official-test cases")
display(primary_features.nlargest(10, "pa_she")[audit_columns].round(4))
print("Low PA-SHE official-test failures")
display(
    primary_features[
        primary_features["rougeL"] < PASHE_PRIMARY_LABEL_THRESHOLD
    ].nsmallest(10, "pa_she")[audit_columns].round(4)
)

## 10. Reporting checklist

- Report the 13 WSI-Bench candidates: Exact plus three thresholds for each of SBERT, BGE, RoBERTa-NLI, and DeBERTa-NLI.
- Report `WSIBENCH_SPLIT_PROTOCOL`. With incomplete training features, validation
  and test are deterministic slide-disjoint derived holdouts, not official splits.
- Report selection at `ROUGE-L < 0.50` using validation AUROC, AUPRC tie-break,
  then candidate-order tie-break across clustering, length alpha and temperature.
- Report separately locked predicted-closed and predicted-open configurations;
  routing uses question text only and excludes the reference answer.
- Evaluate the held-out published-feature test slides only after both rules are locked.
- Report the feature-coverage table and split protocol; do not describe derived
  subset results as official WSI-Bench test results.
- Treat label thresholds `0.30` and `0.70` as sensitivity analyses.
- Report overall, closed-ended and open-ended test AUROC/AUPRC separately.
- Include the raw sequence-log-probability ablation (`alpha=0`, `T=1`).

## 11. Main comparison table

The table uses predicted-closed and predicted-open WSI-Bench clustering and
sequence-weight calibration selected on validation and locked before test
sampling. Utility is shared because all risk methods evaluate the same deployed
greedy predictions. Safety is reported overall, closed-ended and open-ended.

In [ ]:
# Final comparison from validation-locked question-type routing.
import evaluate
from IPython.display import display

comparison_required = [
    "pashe_results", "pashe_open_ended_results", "pashe_routed_test_features",
    "PASHE_LOCKED_CLUSTERING", "PASHE_OPEN_LOCKED_CLUSTERING",
    "PASHE_LOCKED_LENGTH_ALPHA", "PASHE_LOCKED_WEIGHT_TEMPERATURE",
    "PASHE_OPEN_LOCKED_LENGTH_ALPHA", "PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE",
    "PASHE_PRIMARY_LABEL_THRESHOLD",
    "QA_SNNE_LOCKED_VARIANT", "QA_SNNE_LOCKED_METHOD",
    "PASHE_RAW_PA_SHE_METHOD",
    "WSIBENCH_TEST_SCOPE_LABEL",
]
comparison_missing = [
    name for name in comparison_required if name not in globals()
]
if comparison_missing:
    raise RuntimeError(
        "Run PA-SHE sections 5-7 first. Missing: "
        + ", ".join(comparison_missing)
    )

utility_source = pashe_routed_test_features.sort_values(
    "dataset_index"
).drop_duplicates("dataset_index")
references = [
    pashe_normalize(text) for text in utility_source["reference"].astype(str)
]
predictions = [
    pashe_normalize(text) for text in utility_source["prediction"].astype(str)
]
comparison_utility = {
    "BLEU-1": 100.0 * float(evaluate.load("bleu").compute(
        predictions=predictions, references=references, max_order=1
    )["bleu"]),
    "ROUGE-L": 100.0 * float(evaluate.load("rouge").compute(
        predictions=predictions, references=references
    )["rougeL"]),
    "METEOR": 100.0 * float(evaluate.load("meteor").compute(
        predictions=predictions, references=references
    )["meteor"]),
}

overall_safety = pashe_results[
    (pashe_results["evaluation_subset"] == "All")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].copy()
open_safety = pashe_open_ended_results[
    np.isclose(
        pashe_open_ended_results["label_threshold"],
        PASHE_PRIMARY_LABEL_THRESHOLD,
    )
].copy()
closed_safety = pashe_results[
    (pashe_results["evaluation_subset"] == "Closed-ended")
    & np.isclose(pashe_results["label_threshold"], PASHE_PRIMARY_LABEL_THRESHOLD)
].copy()


def comparison_safety(frame, method):
    match = frame[frame["method"] == method]
    if len(match) != 1:
        raise ValueError(f"Expected one locked result for {method}; found {len(match)}")
    row = match.iloc[0]
    return 100.0 * float(row["AUROC"]), 100.0 * float(row["AUPRC"])


locked_variant = (
    f"Closed route={PASHE_LOCKED_CLUSTERING}, alpha={PASHE_LOCKED_LENGTH_ALPHA:g}, "
    f"T={PASHE_LOCKED_WEIGHT_TEMPERATURE:g}; "
    f"Open route={PASHE_OPEN_LOCKED_CLUSTERING}, "
    f"alpha={PASHE_OPEN_LOCKED_LENGTH_ALPHA:g}, "
    f"T={PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE:g}"
)
raw_weighting_variant = (
    "Raw sequence log-probability · alpha=0, T=1; "
    f"Closed route={PASHE_LOCKED_CLUSTERING}; "
    f"Open route={PASHE_OPEN_LOCKED_CLUSTERING}"
)
specification = [
    ("Semantic entropy", locked_variant, "SE"),
    ("Semantic nearest-neighbour entropy", f"ROUGE-L · n={QA_SNNE_NUM_SAMPLES}", "SNNE"),
    ("Visual stability", locked_variant, "VASE"),
    ("PA-SHE ablation", raw_weighting_variant, PASHE_RAW_PA_SHE_METHOD),
    ("Perturbation-Aware Semantic Hallucination Entropy (PA-SHE) (ours)", locked_variant, "PA-SHE"),
]
for qa_variant in QA_SNNE_VARIANTS:
    selected_suffix = (
        " · validation-selected"
        if qa_variant == QA_SNNE_LOCKED_VARIANT else ""
    )
    specification.append((
        "Question-aligned SNNE",
        f"{qa_variant}{selected_suffix} · beta={QA_SNNE_BETA:g}",
        f"QA-SNNE · {qa_variant}",
    ))

rows = []
for family, variant, method in specification:
    overall_auroc, overall_auprc = comparison_safety(overall_safety, method)
    closed_auroc, closed_auprc = comparison_safety(closed_safety, method)
    open_auroc, open_auprc = comparison_safety(open_safety, method)
    rows.append({
        "Uncertainty method": family,
        "Variant / clustering": variant,
        ("Utility", "BLEU-1"): comparison_utility["BLEU-1"],
        ("Utility", "ROUGE-L"): comparison_utility["ROUGE-L"],
        ("Utility", "METEOR"): comparison_utility["METEOR"],
        ("Overall safety", "AUROC"): overall_auroc,
        ("Overall safety", "AUPRC"): overall_auprc,
        ("Closed-ended safety", "AUROC"): closed_auroc,
        ("Closed-ended safety", "AUPRC"): closed_auprc,
        ("Open-ended safety", "AUROC"): open_auroc,
        ("Open-ended safety", "AUPRC"): open_auprc,
    })

comparison_df = pd.DataFrame(rows).set_index([
    "Uncertainty method", "Variant / clustering"
])
comparison_df.columns = pd.MultiIndex.from_tuples(
    comparison_df.columns,
    names=["Evaluation dimension", "Metric"],
)
safety_columns = [
    ("Overall safety", "AUROC"), ("Overall safety", "AUPRC"),
    ("Closed-ended safety", "AUROC"), ("Closed-ended safety", "AUPRC"),
    ("Open-ended safety", "AUROC"), ("Open-ended safety", "AUPRC"),
]
safety_maxima = {column: comparison_df[column].max() for column in safety_columns}


def highlight_maximum(value, column):
    maximum = safety_maxima.get(column)
    if maximum is not None and pd.notna(value) and np.isclose(value, maximum):
        return "font-weight: 700; background-color: #e8f1fb;"
    return ""


comparison_styler = (
    comparison_df.style
    .format("{:.2f}", na_rep="—")
    .apply(
        lambda series: [highlight_maximum(value, series.name) for value in series],
        axis=0,
    )
    .set_caption(
        f"WSI-Bench ({WSIBENCH_TEST_SCOPE_LABEL}): "
        "Validation-Locked Question-Type Routing and Calibration"
    )
    .set_table_styles([
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "18px"),
            ("font-weight", "700"), ("text-align", "left"),
        ]},
        {"selector": "th", "props": [
            ("background-color", "#f5f5f5"), ("border", "1px solid #aaa"),
            ("padding", "8px"), ("text-align", "center"),
        ]},
        {"selector": "td", "props": [
            ("border", "1px solid #b5b5b5"), ("padding", "8px"),
            ("text-align", "center"),
        ]},
        {"selector": "table", "props": [
            ("border-collapse", "collapse"), ("font-size", "13px"),
            ("width", "100%"),
        ]},
    ])
)
display(comparison_styler)
comparison_df.to_csv("WSIBench_PA_SHE_main_comparison_table.csv")
if not wsibench_feature_coverage.empty:
    wsibench_feature_coverage.to_csv(
        "WSIBench_published_feature_coverage.csv", index=False
    )
with open(
    "WSIBench_PA_SHE_main_comparison_table.html",
    "w",
    encoding="utf-8",
) as comparison_file:
    comparison_file.write(comparison_styler.to_html())
print("Evaluation scope:", WSIBENCH_TEST_SCOPE_LABEL)
print("Split protocol:", WSIBENCH_SPLIT_PROTOCOL)
print("Closed-route lock:", PASHE_LOCKED_CLUSTERING, PASHE_LOCKED_LENGTH_ALPHA, PASHE_LOCKED_WEIGHT_TEMPERATURE)
print("Open-route lock:", PASHE_OPEN_LOCKED_CLUSTERING, PASHE_OPEN_LOCKED_LENGTH_ALPHA, PASHE_OPEN_LOCKED_WEIGHT_TEMPERATURE)
print("Saved WSIBench_PA_SHE_main_comparison_table.csv/html")

### Reading the table

- Utility scores describe the same frozen WSI-Bench answer model and therefore
  repeat across uncertainty methods.
- Overall safety uses all selected held-out questions; closed-ended and
  open-ended safety use the annotation-normalized answer types.
- AUROC and AUPRC detect failures defined by `ROUGE-L < 0.50`.
- Semantic rows use validation-locked question-type routing and calibrated
  sequence weights; no configuration is selected on held-out test outcomes.
- Bold cells indicate the best displayed safety ranking only.